# Larger-basis H₂O and N₂ benchmarks: reproduction and extension

This notebook follows the archive note *Larger-Basis H₂O and N₂ Benchmarks* section by section, in the same order as
`larger_basis_full_report.pdf`. Every table puts the archive value next to ours.

| Section | Content |
|---|---|
| 0 | Code (all modules written to files) |
| 1 | Geometries and conventions |
| 2 | Archive Table 1 (orbital counts) and Table 2 (RHF energies) |
| 3 | Retained-orbital windows (which orbitals are kept, per geometry) |
| 4 | Archive Tables 3 and 4 (windowed RHF/CASCI/CASSCF/FCI energies and overlaps), plus the added windows |
| 5 | Basis recommendations: what was done |
| 6 | ADAPT-VQE gradient-measurement costs (new) |
| A/B | Archive Tables 5–18: every orbital energy, archive vs ours |

Heavy runs are switched off (`RUN = False`); results are read from `results/v2/` and `archive_check/`.

## 0. Code

### `windows.py` — window construction (v1) and window FCI

In [1]:
%%writefile windows.py
"""cc-pVDZ retained-orbital windows for H2O (8o) and N2 (11o), following the archive notes.

Orbitals are chosen by irrep at the first geometry and traced to later geometries by
irrep-resolved maximum overlap (archive Sec. 3.3). The window Hamiltonian is the CASCI
effective Hamiltonian of the retained orbitals, so window FCI = CASCI over the window.
"""
import numpy as np
from pyscf import gto, scf, mcscf, fci, ao2mo


def water(r, angle=104.45):
    t = np.deg2rad(angle / 2)
    return f"O 0 0 0; H 0 {r*np.sin(t):.8f} {r*np.cos(t):.8f}; H 0 {-r*np.sin(t):.8f} {r*np.cos(t):.8f}"


def n2(r):
    return f"N 0 0 0; N 0 0 {r}"


def run_rhf(atom, group=True, guess="minao", newton=False):
    """RHF. The archive's stretched N2 solutions are reproduced by a core (1e) guess with
    second-order SCF; H2O uses the default guess."""
    mol = gto.M(atom=atom, basis="cc-pvdz", symmetry=group, unit="Angstrom", verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.max_cycle = 300
    mf.init_guess = guess
    if newton:
        mf = mf.newton()
    mf.kernel()
    assert mf.converged
    return mol, mf


def labels(mol, mf):
    return list(scf.hf_symm.get_orbsym(mol, mf.mo_coeff))


def first_window(system, mol, mf):
    """Pick window orbital indices at the first geometry by irrep (archive Sec. 3.1/3.2)."""
    sym = [mol.irrep_id.index(s) if False else s for s in labels(mol, mf)]
    names = {i: n for i, n in zip(mol.irrep_id, mol.irrep_name)}
    irr = [names[s] for s in sym]
    nocc = mol.nelectron // 2
    occ = list(range(nocc))
    vir = list(range(nocc, len(irr)))

    def lowest(lst, name, k):
        out = [i for i in lst if irr[i] == name][:k]
        assert len(out) == k, (name, k)
        return out

    if system == "H2O":
        core = [0]
        inact_occ = occ[1:]  # 2a1, 1b2, 3a1, 1b1 (all retained)
        # in-plane b irrep = irrep of the lower-energy occupied B orbital
        bocc = [i for i in occ if irr[i] != "A1"]
        b_in = irr[bocc[0]]
        virt = lowest(vir, "A1", 2) + lowest(vir, b_in, 2)  # 4a1,5a1 ; 2b2,3b2
    else:  # N2, D2h
        core = [0, 1]
        inact_occ = occ[2:]  # 2sg, 2su, 1pu x2, 3sg
        virt = (lowest(vir, "B2g", 1) + lowest(vir, "B3g", 1) + lowest(vir, "B1u", 1)
                + lowest(vir, "Ag", 1) + lowest(vir, "B2u", 1) + lowest(vir, "B3u", 1))
    return core, sorted(inact_occ), sorted(virt), irr


def trace(prev, mol, mf, sel):
    """Map previous-geometry orbitals to current ones: same irrep, same occ/vir class, max overlap."""
    pmol, pmf = prev
    S = gto.intor_cross("int1e_ovlp", pmol, mol)
    O = np.abs(pmf.mo_coeff.T @ S @ mf.mo_coeff)
    pirr = [dict(zip(pmol.irrep_id, pmol.irrep_name))[s] for s in labels(pmol, pmf)]
    irr = [dict(zip(mol.irrep_id, mol.irrep_name))[s] for s in labels(mol, mf)]
    nocc = mol.nelectron // 2
    out, used = [], set()
    for p in sel:
        cls = range(nocc) if p < nocc else range(nocc, len(irr))
        cand = [i for i in cls if irr[i] == pirr[p] and i not in used]
        best = max(cand, key=lambda i: O[p, i])
        out.append(best); used.add(best)
    return out, irr


def by_counts(mol, mf, ncore, counts, prev=None, prev_sel=None):
    """Keep the core frozen, retain every other occupied orbital, and fill each irrep up to its
    equilibrium count with the lowest virtuals of that irrep (keeps degenerate pairs together)."""
    irr = [dict(zip(mol.irrep_id, mol.irrep_name))[s] for s in labels(mol, mf)]
    nocc = mol.nelectron // 2
    core = list(range(ncore))
    occ = list(range(ncore, nocc))
    left = dict(counts)
    for i in occ:
        left[irr[i]] -= 1
    vir = []
    if prev is not None:  # projection of each virtual onto the previous retained space
        pmol, pmf = prev
        S = gto.intor_cross("int1e_ovlp", pmol, mol)
        O = pmf.mo_coeff[:, prev_sel].T @ S @ mf.mo_coeff
        score = (O ** 2).sum(0)
    for name, k in left.items():
        assert k >= 0, (name, k)
        cand = [i for i in range(nocc, len(irr)) if irr[i] == name]
        if prev is not None:
            cand = sorted(cand, key=lambda i: -score[i])
        vir += cand[:k]
    return core + occ + sorted(vir), irr


def window_hamiltonian(mol, mf, core, win):
    ncas = len(win)
    nelec = mol.nelectron - 2 * len(core)
    mc = mcscf.CASCI(mf, ncas, nelec)
    order = list(core) + list(win) + [i for i in range(mf.mo_coeff.shape[1]) if i not in core and i not in win]
    mo = mf.mo_coeff[:, order]
    h1, ecore = mc.get_h1eff(mo)
    eri = ao2mo.restore(1, mc.get_h2eff(mo), ncas)
    e_fci, civec = fci.direct_spin1.kernel(h1, eri, ncas, nelec, ecore=ecore, conv_tol=1e-12, max_cycle=300, nroots=1)
    return dict(h1=h1, eri=eri, ecore=ecore, ncas=ncas, nelec=nelec, e_fci=e_fci, civec=civec)


SYSTEMS = {
    "H2O": dict(geoms=[0.9584, 1.75, 3.0], atom=water, group=True, guess="minao", newton=False,
                ref=[(-76.026741428, -76.059281484), (-75.669624909, -75.826835961), (-75.430561431, -75.732177025)]),
    "N2": dict(geoms=[1.10, 1.80, 3.00], atom=n2, group="D2h", guess="1e", newton=True,
               ref=[(-108.953796253, -109.059444517), (-108.341640217, -108.806606966), (-108.231992299, -108.749118387)]),
}


def build_all(system):
    spec = SYSTEMS[system]
    out, prev, sel = [], None, None
    for k, r in enumerate(spec["geoms"]):
        mol, mf = run_rhf(spec["atom"](r), spec["group"], spec["guess"], spec["newton"])
        if prev is None:
            core, occ, vir, irr = first_window(system, mol, mf)
            sel = core + occ + vir
            new = sel
            counts = {}
            for i in occ + vir:
                counts[irr[i]] = counts.get(irr[i], 0) + 1
        else:
            new, irr = by_counts(mol, mf, len(core), counts, prev, sel[len(core):])
        ncore = 1 if system == "H2O" else 2
        core, win = new[:ncore], new[ncore:]
        nocc = mol.nelectron // 2
        win = sorted([i for i in win if i < nocc]) + sorted([i for i in win if i >= nocc])
        W = window_hamiltonian(mol, mf, core, win)
        W.update(system=system, R=r, e_hf=mf.e_tot, mol=mol, mf=mf, core=core, win=win,
                 win_irreps=[irr[i] for i in win], mo_energy=mf.mo_energy[win])
        W["ref_hf"], W["ref_fci"] = spec["ref"][k]
        out.append(W)
        prev, sel = (mol, mf), new
    return out


if __name__ == "__main__":
    import sys
    for s in sys.argv[1:] or ["H2O", "N2"]:
        for W in build_all(s):
            print(f"{s} R={W['R']:.4f} win={W['win_irreps']} "
                  f"dHF={1e3*(W['e_hf']-W['ref_hf']):+.4f} mHa dFCI={1e3*(W['e_fci']-W['ref_fci']):+.4f} mHa", flush=True)


Overwriting windows.py


### `windows2.py` — general windows, CAS states, overlaps

In [2]:
%%writefile windows2.py
"""General retained-orbital windows (archive Sec. 3) for several bases and window sizes.

A case is (system, basis, window). At the first geometry the window is chosen by irrep:
core frozen, every other occupied orbital kept, and a fixed number of the lowest virtuals of
each irrep. At later geometries the core stays frozen, all occupied orbitals are kept, and each
irrep is filled back to the same count with the virtuals that overlap most with the previous
window (archive Sec. 3.3). A cc-pVTZ window takes its virtuals by overlap with the cc-pVDZ window
at the same geometry (character-defined window, archive Sec. 5).
"""
import numpy as np
from pyscf import gto, scf, mcscf, fci, ao2mo
from pyscf.scf import addons as scf_addons
from windows import water, n2, labels, window_hamiltonian

# virtual orbitals kept per irrep at the first geometry ("Bin" = in-plane b of H2O, "Bout" = out-of-plane b)
VIRT = {
    ("H2O", "8o"): {"A1": 2, "Bin": 2},
    ("H2O", "11o"): {"A1": 3, "Bin": 3, "Bout": 1},
    ("H2O", "14o"): {"A1": 4, "Bin": 3, "Bout": 2, "A2": 1},
    ("N2", "11o"): {"Ag": 1, "B1u": 1, "B2u": 1, "B3u": 1, "B2g": 1, "B3g": 1},
}
NCORE = {"H2O": 1, "N2": 2}

CASES = {
    # key: (system, basis, window, geometries, (guess, newton), archive (E_RHF, E_FCI) per geometry or None)
    "H2O_ccpvdz_8o": ("H2O", "cc-pvdz", "8o", [0.9584, 1.75], ("minao", False),
                      [(-76.026741428, -76.059281484), (-75.669624909, -75.826835961)]),
    "H2O_ccpvdz_11o": ("H2O", "cc-pvdz", "11o", [0.9584, 1.75], ("minao", False), None),
    "H2O_ccpvdz_14o": ("H2O", "cc-pvdz", "14o", [0.9584, 1.75], ("minao", False), None),
    "N2_ccpvdz_11o": ("N2", "cc-pvdz", "11o", [1.10, 1.80, 3.00], ("1e", True),
                      [(-108.953796253, -109.059444517), (-108.341640217, -108.806606966), (-108.231992299, -108.749118387)]),
    "H2O_631g_all": ("H2O", "6-31g", "all", [0.9584, 1.75, 3.0], ("huckel", False),
                     [(-75.983953320, None), (-75.653868837, None), (-75.316037599, None)]),
    "H2O_ccpvtz_8o": ("H2O", "cc-pvtz", "8o", [1.75], None, None),
    "N2_ccpvtz_11o": ("N2", "cc-pvtz", "11o", [1.80], None, None),
}

# archive CASCI / CASSCF energies (Table 3), used to fix the CAS orbitals
ARCHIVE_CAS = {
    ("H2O", 0.9584): (-76.030123418, -76.054064392), ("H2O", 1.75): (-75.799299906, -75.824182389),
    ("N2", 1.10): (-109.021904308, -109.023889473), ("N2", 1.80): (-108.757342076, -108.777835125),
    ("N2", 3.00): (-108.695809945, -108.738528934),
}
CAS_SIZE = {"H2O": (4, 4), "N2": (6, 6)}          # (orbitals, electrons)


def geom(system, r):
    return water(r) if system == "H2O" else n2(r)


def mk_mol(system, basis, r):
    return gto.M(atom=geom(system, r), basis=basis, symmetry=True if system == "H2O" else "D2h",
                 unit="Angstrom", verbose=0)


def rhf(mol, guess="minao", newton=False, dm0=None):
    mf = scf.RHF(mol); mf.conv_tol = 1e-11; mf.max_cycle = 300; mf.init_guess = guess
    if newton:
        mf = mf.newton()
    mf.kernel(dm0)
    assert mf.converged
    return mf


def irrep_names(mol, mf):
    d = dict(zip(mol.irrep_id, mol.irrep_name))
    return [d[s] for s in labels(mol, mf)]


def h2o_inplane_b(mol):
    """The B irrep that contains hydrogen AOs (in-plane O-H bonding/antibonding)."""
    hs = [i for i, lab in enumerate(mol.ao_labels()) if lab.split()[1] == "H" and lab.split()[2].endswith("s")]  # H s AOs only
    for name, so in zip(mol.irrep_name, mol.symm_orb):
        if name.startswith("B") and np.abs(so[hs]).max() > 1e-8:
            return name
    raise RuntimeError("no in-plane B irrep")


def resolve(system, irr, nocc, counts, mol=None):
    if system != "H2O":
        return dict(counts)
    b_in = h2o_inplane_b(mol)
    b_out = "B1" if b_in == "B2" else "B2"
    out = {}
    for k, v in counts.items():
        out[{"Bin": b_in, "Bout": b_out}.get(k, k)] = v
    return out


def pick(mol, mf, ncore, counts_total, ref=None, ref_sel=None):
    """core + all occupied + virtuals per irrep (lowest, or by overlap with a reference window)."""
    irr = irrep_names(mol, mf)
    nocc = mol.nelectron // 2
    occ = list(range(ncore, nocc))
    left = dict(counts_total)
    for i in occ:
        left[irr[i]] = left.get(irr[i], 0) - 1
    if ref is not None:
        rmol, rmf = ref
        S = gto.intor_cross("int1e_ovlp", rmol, mol)
        O = rmf.mo_coeff[:, ref_sel].T @ S @ mf.mo_coeff
        score = (O ** 2).sum(0)
    vir = []
    for name, k in left.items():
        assert k >= 0, (name, k)
        cand = [i for i in range(nocc, len(irr)) if irr[i] == name]
        if ref is not None:
            cand = sorted(cand, key=lambda i: -score[i])
        vir += cand[:k]
    return list(range(ncore)), occ, sorted(vir), irr


def build_case(key, fci_solve=True):
    system, basis, window, geoms, rhfopt, refs = CASES[key]
    ncore = NCORE[system]
    out = []
    if basis == "cc-pvtz":
        dz = {w["R"]: w for w in build_case(f"{system}_ccpvdz_{window}", fci_solve=False)}
    prev = None
    for k, r in enumerate(geoms):
        mol = mk_mol(system, basis, r)
        if basis == "cc-pvtz":
            W0 = dz[r]
            dm0 = scf_addons.project_dm_nr2nr(W0["mol"], W0["mf"].make_rdm1(), mol)
            mf = rhf(mol, dm0=dm0)
            counts = {}
            for n in W0["win_irreps"]:
                counts[n] = counts.get(n, 0) + 1
            core, occ, vir, irr = pick(mol, mf, ncore, counts, (W0["mol"], W0["mf"]), W0["win"])
        else:
            mf = rhf(mol, *rhfopt)
            if window == "all":
                irr = irrep_names(mol, mf); nocc = mol.nelectron // 2
                core, occ, vir = list(range(ncore)), list(range(ncore, nocc)), list(range(nocc, len(irr)))
            elif prev is None:
                irr = irrep_names(mol, mf); nocc = mol.nelectron // 2
                virt = resolve(system, irr, nocc, VIRT[(system, window)], mol)
                counts = {}
                for i in range(ncore, nocc):
                    counts[irr[i]] = counts.get(irr[i], 0) + 1
                for n, v in virt.items():
                    counts[n] = counts.get(n, 0) + v
                core, occ, vir, irr = pick(mol, mf, ncore, counts)
            else:
                core, occ, vir, irr = pick(mol, mf, ncore, counts, prev, prev_sel)
        win = occ + vir
        if fci_solve:
            W = window_hamiltonian(mol, mf, core, win)
        else:
            W = dict(ncas=len(win), nelec=mol.nelectron - 2 * len(core))
        W.update(key=key, system=system, basis=basis, window=window, R=r, e_hf=mf.e_tot, mol=mol, mf=mf,
                 core=core, win=win, win_irreps=[irr[i] for i in win])
        W["ref_hf"], W["ref_fci"] = (refs[k] if refs else (None, None))
        out.append(W)
        prev, prev_sel = (mol, mf), win
    return out


# ------------------------------------------------------------------ CAS model space

def window_mf(W):
    """RHF object living in the window only (orbitals = the RHF window orbitals)."""
    norb, nelec = W["ncas"], W["nelec"]
    mol = gto.M(verbose=0); mol.nelectron = nelec; mol.incore_anyway = True
    mol.nao_nr = lambda *a, **k: norb
    mf = scf.RHF(mol)
    mf.get_hcore = lambda *a: W["h1"]
    mf.get_ovlp = lambda *a: np.eye(norb)
    mf._eri = ao2mo.restore(8, W["eri"], norb)
    mf.energy_nuc = lambda *a: W["ecore"]
    dm = np.zeros((norb, norb)); dm[range(nelec // 2), range(nelec // 2)] = 2
    mf.kernel(dm)
    return mf


def cas_candidates(W):
    """All irrep-consistent CAS choices inside the window (small search)."""
    import itertools
    system = W["system"]; irr = W["win_irreps"]; nocc_w = W["nelec"] // 2
    ncas, ne = CAS_SIZE[system]
    occ = list(range(nocc_w)); vir = list(range(nocc_w, W["ncas"]))
    nact_occ = ne // 2
    for a in itertools.combinations(occ, nact_occ):
        for b in itertools.combinations(vir, ncas - nact_occ):
            yield list(a) + list(b)


def casci_energy(W, wmf, cas):
    ncas, ne = CAS_SIZE[W["system"]]
    mc = mcscf.CASCI(wmf, ncas, ne)
    mo = mc.sort_mo([c + 1 for c in cas], mo_coeff=np.eye(W["ncas"]))
    return mc.kernel(mo)[0]


def choose_cas(W, wmf, ref_cas=None):
    """CAS orbitals (window indices). With archive energies: the choice whose CASCI matches Table 3.
    Otherwise by maximum overlap with a reference CAS given as (mol, AO coefficients)."""
    if ref_cas is None:
        target = ARCHIVE_CAS[(W["system"], round(W["R"], 4))][0]
        irr = W["win_irreps"]
        # CAS irreps are fixed by chemistry; only try choices whose irreps fit (A1/B for H2O, one per irrep type for N2)
        best = None
        for cas in cas_candidates(W):
            e = casci_energy(W, wmf, cas)
            if best is None or abs(e - target) < abs(best[1] - target):
                best = (cas, e)
            if abs(e - target) < 1e-6:
                break
        return best
    rmol, rC = ref_cas
    S = gto.intor_cross("int1e_ovlp", rmol, W["mol"])
    Cw = W["mf"].mo_coeff[:, W["win"]]
    O = np.abs(rC.T @ S @ Cw)
    cas = []
    for row in O:
        order = [i for i in np.argsort(-row) if i not in cas]
        cas.append(int(order[0]))
    return sorted(cas), casci_energy(W, wmf, sorted(cas))


def cas_states(W, cas):
    """CASCI (RHF frame) and CASSCF as window-FCI vectors in the RHF window frame.
    CASSCF runs on the real molecule with point-group symmetry; every orbital outside the window
    (frozen core and external virtuals) is frozen, so the orbitals are optimised inside the window only."""
    from pyscf.fci import cistring
    mol, mf = W["mol"], W["mf"]
    norb, nelec = W["ncas"], W["nelec"]
    ncas, ne = CAS_SIZE[W["system"]]
    S = mol.intor("int1e_ovlp")
    Cw = mf.mo_coeff[:, W["win"]]
    caslst = [W["win"][c] for c in cas]
    out = {}
    for kind in ("CASCI", "CASSCF"):
        mc = (mcscf.CASCI if kind == "CASCI" else mcscf.CASSCF)(mf, ncas, ne)
        mc.canonicalization = False   # canonicalization would mix window and external orbitals
        mc.natorb = False
        mc.fcisolver.wfnsym = "A1" if W["system"] == "H2O" else "Ag"   # totally symmetric singlet root
        mc.fix_spin_(ss=0)
        ncore_t = mc.ncore
        noncas = [i for i in range(mf.mo_coeff.shape[1]) if i not in caslst]
        order = noncas[:ncore_t] + caslst + noncas[ncore_t:]
        mo = mf.mo_coeff[:, order]
        inwin = set(W["win"])
        if kind == "CASSCF":
            mc.frozen = [p for p, i in enumerate(order) if i not in inwin]
            mc.conv_tol = 1e-10
        mc.kernel(mo)
        pos = [p for p, i in enumerate(order) if i in inwin]
        C = Cw.T @ S @ mc.mo_coeff[:, pos]              # new window orbitals in the RHF window basis
        assert np.abs(C.T @ C - np.eye(norb)).max() < 1e-8
        ncore = ncore_t - len(W["core"])
        na = ne // 2
        sa_full = cistring.make_strings(range(norb), nelec // 2)
        addr = {int(x): i for i, x in enumerate(sa_full)}
        sa_cas = cistring.make_strings(range(ncas), na)
        corebits = (1 << ncore) - 1
        ci = np.asarray(mc.ci).reshape(len(sa_cas), len(sa_cas))
        full = np.zeros((len(sa_full), len(sa_full)))
        for i, a in enumerate(sa_cas):
            for j, b in enumerate(sa_cas):
                full[addr[corebits | (int(a) << ncore)], addr[corebits | (int(b) << ncore)]] = ci[i, j]
        vec = fci.addons.transform_ci(full, (nelec // 2, nelec // 2), C.T)
        e = fci.direct_spin1.energy(W["h1"], W["eri"], vec, norb, nelec) + W["ecore"]
        out[kind] = dict(vec=vec, e=e, e_mc=mc.e_tot, C=C, ncore=ncore)
    return out


def overlaps(W, cs):
    """Archive Table 4 quantities: |<HF|FCI>|^2, |<CASSCF|FCI>|^2, FCI weight in the CASSCF model space."""
    from pyscf.fci import cistring
    f = W["civec"]; norb, nelec = W["ncas"], W["nelec"]
    hf_fci = f[0, 0] ** 2
    cas_fci = float(np.sum(cs["CASSCF"]["vec"] * f)) ** 2
    C, ncore = cs["CASSCF"]["C"], cs["CASSCF"]["ncore"]
    ncas = CAS_SIZE[W["system"]][0]
    fr = fci.addons.transform_ci(f, (nelec // 2, nelec // 2), C)   # FCI in CASSCF orbitals
    s = cistring.make_strings(range(norb), nelec // 2)
    core = (1 << ncore) - 1
    ok = np.array([(int(x) & core) == core and (int(x) >> (ncore + ncas)) == 0 for x in s])
    w_cas = float((fr[np.ix_(ok, ok)] ** 2).sum())
    return hf_fci, cas_fci, w_cas


Overwriting windows2.py


### `paulis.py` — Pauli algebra, Hamiltonian, UCCSD pool, gradient operators

In [3]:
%%writefile paulis.py
"""Pauli algebra in (x, z) bit-mask form, JW Hamiltonian, UCCSD pool and gradient operators.

Conventions follow the Part I notebook: qubit q is bit (nq-1-q); spin orbital 2p = alpha, 2p+1 = beta;
P(x, z) = i^{|x&z|} X^x Z^z, so Y = iXZ.
"""
import itertools
import numpy as np
import scipy.sparse as sp
import openfermion as of
from openfermion.chem.molecular_data import spinorb_from_spatial

U = np.uint64


def pc(a):
    return np.bitwise_count(np.asarray(a, dtype=U)).astype(np.int64)


def term_to_xz(term, nq):
    x = z = 0
    for q, p in term:
        bit = 1 << (nq - 1 - q)
        if p in "XY":
            x |= bit
        if p in "ZY":
            z |= bit
    return x, z


def qop_to_arrays(op, nq, tol=1e-12):
    xs, zs, cs = [], [], []
    for t, c in op.terms.items():
        if abs(c) < tol or len(t) == 0 and False:
            continue
        x, z = term_to_xz(t, nq)
        xs.append(x); zs.append(z); cs.append(c)
    return np.array(xs, dtype=U), np.array(zs, dtype=U), np.array(cs, dtype=complex)


def hamiltonian(W):
    h1, eri, ecore = W["h1"], W["eri"], W["ecore"]
    two = np.asarray(eri.transpose(0, 2, 3, 1), order="C")
    one_so, two_so = spinorb_from_spatial(h1, two)
    iop = of.InteractionOperator(ecore, one_so, 0.5 * two_so)
    H = of.jordan_wigner(of.get_fermion_operator(iop))
    H.compress(1e-10)
    nq = 2 * h1.shape[0]
    x, z, c = qop_to_arrays(H, nq)
    return nq, x, z, c.real


def uccsd_pool(nq, nelec):
    occ = list(range(nelec)); vir = list(range(nelec, nq))
    pool, labels = [], []
    for i in occ:
        for a in vir:
            if i % 2 == a % 2:
                op = of.FermionOperator(((a, 1), (i, 0))) - of.FermionOperator(((i, 1), (a, 0)))
                pool.append(op); labels.append(f"S {i}->{a}")
    for i, j in itertools.combinations(occ, 2):
        for a, b in itertools.combinations(vir, 2):
            if (i % 2 + j % 2) == (a % 2 + b % 2):
                if {i % 2, j % 2} != {a % 2, b % 2} and (i % 2 + j % 2) == 1:
                    continue
                op = of.FermionOperator(((b, 1), (a, 1), (j, 0), (i, 0)))
                op = op - of.hermitian_conjugated(op)
                pool.append(op); labels.append(f"D {i},{j}->{a},{b}")
    out = []
    for op in pool:
        q = of.jordan_wigner(op); q.compress(1e-12)
        out.append(qop_to_arrays(q, nq))
    return out, labels


def mult(x1, z1, x2, z2):
    """P1*P2 = i^k P3 with k returned mod 4 (elementwise, broadcast)."""
    x3, z3 = x1 ^ x2, z1 ^ z2
    k = pc(x1 & z1) + pc(x2 & z2) - pc(x3 & z3) + 2 * pc(z1 & x2)
    return x3, z3, np.mod(k, 4)


IK = np.array([1, 1j, -1, -1j])


def commutator(Hx, Hz, Hc, Gx, Gz, Gc):
    """[H, G] for H Hermitian (real coeffs) and G anti-Hermitian (imag coeffs). Returns real coeffs."""
    X, Z, C = [], [], []
    for gx, gz, gc in zip(Gx, Gz, Gc):
        anti = (pc((Hx & gz) ^ (Hz & gx)) & 1).astype(bool)
        hx, hz, hc = Hx[anti], Hz[anti], Hc[anti]
        x3, z3, k = mult(hx, hz, gx, gz)
        X.append(x3); Z.append(z3); C.append(2 * hc * gc * IK[k])
    X = np.concatenate(X); Z = np.concatenate(Z); C = np.concatenate(C)
    key = (X << U(32)) | Z
    uk, inv = np.unique(key, return_inverse=True)
    cr = np.bincount(inv, weights=C.real, minlength=len(uk))
    ci = np.bincount(inv, weights=C.imag, minlength=len(uk))
    assert np.abs(ci).max(initial=0) < 1e-9
    keep = np.abs(cr) > 1e-10
    return uk[keep], cr[keep]


def gradient_matrix(nq, Hx, Hz, Hc, pool):
    keys, coefs = [], []
    for gx, gz, gc in pool:
        k, c = commutator(Hx, Hz, Hc, gx, gz, gc)
        keys.append(k); coefs.append(c)
    allk = np.unique(np.concatenate(keys))
    rows, cols, vals = [], [], []
    for i, (k, c) in enumerate(zip(keys, coefs)):
        rows.append(np.full(len(k), i)); cols.append(np.searchsorted(allk, k)); vals.append(c)
    A = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(len(pool), len(allk)))
    xs = allk >> U(32); zs = allk & U(0xFFFFFFFF)
    return A, xs.astype(U), zs.astype(U)


def label(x, z, nq):
    s = []
    for q in range(nq):
        b = 1 << (nq - 1 - q)
        s.append({(0, 0): "I", (1, 0): "X", (0, 1): "Z", (1, 1): "Y"}[(bool(int(x) & b), bool(int(z) & b))])
    return "".join(s)


Overwriting paulis.py


### `states.py` — HF and CISD states

In [4]:
%%writefile states.py
"""HF, CISD and FCI states of a window, as sparse JW bit-string vectors."""
import numpy as np
import scipy.sparse.linalg as sla
from pyscf import fci
from pyscf.fci import cistring
from meas import State
from paulis import U, pc


def to_jw(civec, norb, nelec, tol=1e-10):
    """pyscf (alpha-string, beta-string) CI vector -> JW basis states.
    pyscf orders a+_alpha... a+_beta...; JW/OpenFermion orders by spin-orbital index
    (2k = alpha, 2k+1 = beta). Reordering sign = (-1)^{#(beta j, alpha k) with j < k}."""
    na = nb = nelec // 2
    sa = cistring.make_strings(range(norb), na)
    sb = cistring.make_strings(range(norb), nb)
    nq = 2 * norb
    ia, ib = np.nonzero(np.abs(civec) > tol)
    idx = np.zeros(len(ia), dtype=U); sign = np.ones(len(ia))
    for n, (i, j) in enumerate(zip(ia, ib)):
        a, b = int(sa[i]), int(sb[j])
        bits = 0
        for k in range(norb):
            if a >> k & 1:
                bits |= 1 << (nq - 1 - 2 * k)
            if b >> k & 1:
                bits |= 1 << (nq - 1 - (2 * k + 1))
        cross = 0
        for k in range(norb):
            if a >> k & 1:
                cross += bin(b & ((1 << k) - 1)).count("1")
        idx[n] = bits; sign[n] = -1 if cross & 1 else 1
    return State(idx, civec[ia, ib] * sign)


def hf_state(nq, nelec):
    return State(np.array([sum(1 << (nq - 1 - q) for q in range(nelec))], dtype=U), np.array([1.0]))


def cisd_vector(W, spin_penalty=1.0):
    """Lowest singlet in the space of the HF determinant plus all single and double excitations."""
    norb, nelec = W["ncas"], W["nelec"]
    na = nelec // 2
    sa = cistring.make_strings(range(norb), na)
    hf = (1 << na) - 1
    exc = np.array([bin(int(s) & ~hf).count("1") for s in sa])
    mask = (exc[:, None] + exc[None, :]) <= 2
    sel = np.flatnonzero(mask.ravel())
    shape = (len(sa), len(sa))
    h2e = fci.direct_spin1.absorb_h1e(W["h1"], W["eri"], norb, nelec, 0.5)

    def mv(v):
        full = np.zeros(shape[0] * shape[1]); full[sel] = v
        full = full.reshape(shape)
        out = fci.direct_spin1.contract_2e(h2e, full, norb, nelec)
        out = out + spin_penalty * fci.spin_op.contract_ss(full, norb, nelec)
        return out.ravel()[sel]

    op = sla.LinearOperator((len(sel), len(sel)), matvec=mv, dtype=float)
    e, v = sla.eigsh(op, k=1, which="SA", tol=1e-12)
    full = np.zeros(shape[0] * shape[1]); full[sel] = v[:, 0]
    full = full.reshape(shape)
    e_cisd = fci.direct_spin1.energy(W["h1"], W["eri"], full, norb, nelec) + W["ecore"]
    return full, e_cisd, len(sel)


def energy(state, Hx, Hz, Hc):
    from meas import pauli_expect
    return float(Hc @ pauli_expect(state, Hx, Hz))


Overwriting states.py


### `meas.py` — FC grouping, fragment variances, Part I methods

In [5]:
%%writefile meas.py
"""States, fast FC grouping, fragment variances and the Part I oracle methods (M1, M2, M3, M3-R).

The method code (allocate, m1/m2/m3/m3r costs) is copied from the Part I notebook; only the
data structures changed (sparse A, sparse states) so that 16- and 22-qubit windows fit in memory.
"""
import numpy as np
import scipy.sparse as sp
from scipy.stats import norm
from paulis import U, pc

# ---------------------------------------------------------------- grouping

def group_fc(xs, zs, order, nq):
    """First-fit sorted insertion into fully commuting groups (same result as the Part I
    routine). A Pauli commutes with every member of a group iff it commutes with a GF(2) basis
    of the group, so only the basis (<= nq vectors per group) is stored and checked."""
    L = len(xs)
    gid = -np.ones(L, dtype=np.int64)
    cap = 1 << 16
    bx = np.zeros(cap, dtype=U); bz = np.zeros(cap, dtype=U); bg = np.zeros(cap, dtype=np.int64)
    nb = 0
    piv = []  # per group: dict pivot -> (vector)
    ng = 0
    for l in order:
        x, z = xs[l], zs[l]
        if nb:
            bad = (pc((bx[:nb] & z) ^ (bz[:nb] & x)) & 1).astype(bool)
            badg = np.bincount(bg[:nb][bad], minlength=ng)
            ok = np.flatnonzero(badg == 0)
            g = int(ok[0]) if len(ok) else ng
        else:
            g = 0
        if g == ng:
            ng += 1; piv.append({})
        gid[l] = g
        v = (int(x) << 32) | int(z)
        rows = piv[g]
        for p, r in rows.items():
            if v >> p & 1:
                v ^= r
        if v:
            p = v.bit_length() - 1
            for q in list(rows):
                if rows[q] >> p & 1:
                    rows[q] ^= v
            rows[p] = v
            if nb == cap:
                cap *= 2
                bx = np.resize(bx, cap); bz = np.resize(bz, cap); bg = np.resize(bg, cap)
            bx[nb] = U(v >> 32); bz[nb] = U(v & 0xFFFFFFFF); bg[nb] = g; nb += 1
    return gid, ng


def base_partition(A, xs, zs, nq):
    w = np.round(np.sqrt(np.asarray(A.multiply(A).sum(0)).ravel()), 10)   # rounding makes ties deterministic
    gid, ng = group_fc(xs, zs, np.argsort(-w, kind="stable"), nq)
    return split(gid, ng)


def split(gid, ng):
    order = np.argsort(gid, kind="stable")
    cuts = np.searchsorted(gid[order], np.arange(ng + 1))
    return [order[cuts[k]:cuts[k + 1]] for k in range(ng)]


# ---------------------------------------------------------------- sparse states

class State:
    """psi = sum_b amp[b] |b>, with basis states stored as sorted uint64 bit strings."""

    def __init__(self, idx, amp):
        o = np.argsort(idx)
        self.idx = np.asarray(idx, dtype=U)[o]
        self.amp = np.asarray(amp, dtype=complex)[o]
        self.amp /= np.linalg.norm(self.amp)

    def lookup(self, keys):
        pos = np.searchsorted(self.idx, keys)
        pos = np.minimum(pos, len(self.idx) - 1)
        hit = self.idx[pos] == keys
        out = np.zeros(len(keys), dtype=complex)
        out[hit] = self.amp[pos[hit]]
        return out


def pauli_expect(state, xs, zs):
    """<psi|P_l|psi> for many Paulis."""
    out = np.zeros(len(xs))
    ny = pc(xs & zs)
    for b, a in zip(state.idx, state.amp):
        sgn = 1 - 2 * (pc(b & zs) & 1)
        out += (np.conj(state.lookup(b ^ xs)) * a * sgn * (1j) ** ny).real
    return out


def fragment_variances(state, A, xs, zs, groups, batch_pairs=1_000_000):
    """q[i, a] = Var_psi(F_ia), mu[i, a] = <F_ia>, F_ia = sum_{l in group a} A[i, l] P_l.

    F_ia psi = sum_l A_il sum_b psi_b phase(l, b) |b ^ x_l>. Every (Pauli l, det b) pair is sent to
    the bucket (group a, output state b ^ x_l); a sparse product with A then gives all amplitudes."""
    A = sp.csc_matrix(A)
    K, G = A.shape[0], len(groups)
    q = np.zeros((K, G)); mu = np.zeros((K, G))
    nd = len(state.idx)
    per = max(1, batch_pairs // nd)
    gi = 0
    while gi < G:
        # take groups until the pair budget is used
        sel, npau = [], 0
        while gi < G and (npau == 0 or npau + len(groups[gi]) <= per):
            sel.append(gi); npau += len(groups[gi]); gi += 1
        mem = np.concatenate([groups[g] for g in sel])
        gof = np.concatenate([np.full(len(groups[g]), k) for k, g in enumerate(sel)])
        x = xs[mem]; z = zs[mem]; ny = pc(x & z)
        out = (x[:, None] ^ state.idx[None, :])                      # (npau, nd)
        sgn = 1 - 2 * (pc(state.idx[None, :] & z[:, None]) & 1)
        val = sgn * (1j) ** ny[:, None] * state.amp[None, :]
        key = (gof[:, None].astype(U) << U(40)) ^ out                 # groups < 2^23, states < 2^40
        uk, inv = np.unique(key.ravel(), return_inverse=True)
        Phi = sp.csr_matrix((val.ravel(), (np.repeat(np.arange(len(mem)), nd), inv)), shape=(len(mem), len(uk)))
        M = (A[:, mem] @ Phi).tocsr()                                 # K x buckets, amplitudes of F psi
        bgrp = (uk >> U(40)).astype(np.int64)
        bout = uk & U((1 << 40) - 1)
        cpsi = np.conj(state.lookup(bout))
        Mc = M.tocoo()
        norm2 = np.zeros((K, len(sel))); mean = np.zeros((K, len(sel)))
        np.add.at(norm2, (Mc.row, bgrp[Mc.col]), np.abs(Mc.data) ** 2)
        np.add.at(mean, (Mc.row, bgrp[Mc.col]), (cpsi[Mc.col] * Mc.data).real)
        q[:, sel] = np.maximum(norm2 - mean ** 2, 0.0)
        mu[:, sel] = mean
    return q, mu


# ---------------------------------------------------------------- Part I methods (copied)

def z_value(n_arms, delta=0.05):
    return norm.ppf(1 - delta / (2 * n_arms))


def allocate(q, eps, iters=400, tol=1e-9):
    q = np.asarray(q, float)
    need = q.sum(0) > 0
    q = q[:, need]
    eps2 = np.asarray(eps, float) ** 2
    lam = np.ones(q.shape[0])
    best = None
    stall = 0
    for it in range(iters):
        n = np.sqrt(np.maximum(lam @ q, 1e-300))
        V = (q / n).sum(1)
        scale = np.max(V / eps2)
        cost = scale * n.sum()
        if best is None or cost < best[0] - tol * best[0]:
            best = (cost, n * scale); stall = 0
        else:
            stall += 1
            if stall > 60:      # no improvement for 60 steps: converged
                break
        lam = lam * np.sqrt(V / eps2)
        lam /= lam.max()
    out = np.zeros(len(need))
    out[need] = best[1]
    return best[0], out


def gaps(g, delta=0.0):
    """delta > 0 gives the Delta-good target: arms within delta of the best need not be separated,
    so every elimination radius is at least delta/2."""
    a = np.abs(g)
    w = int(np.argmax(a))
    s = np.sort(a)[::-1]
    gap = max(s[0] - s[1], delta)
    D = np.maximum(a[w] - a, delta)
    D[w] = gap
    return w, gap, D


def m1_cost(q, g, z, delta=0.0):
    w, gap, D = gaps(g, delta)
    eps = np.full(q.shape[0], gap / 2 / z)
    active = q.sum(1) > 0
    return allocate(q[active], eps[active])


def m2_cost(q_list, g, z, delta=0.0):
    w, gap, D = gaps(g, delta)
    per = np.zeros(len(g))
    for i, qi in enumerate(q_list):
        s = np.sqrt(qi).sum()
        if s == 0:
            continue
        per[i] = s ** 2 / (D[i] / 2 / z) ** 2
    return per.sum(), per


def m3_cost(q, g, z, delta=0.0):
    w, gap, D = gaps(g, delta)
    levels = np.unique(np.round(D, 12))[::-1]
    charged = np.zeros(q.shape[1])
    stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (q.sum(1) > 0))
        if len(act) == 0:
            continue
        cost, n = allocate(q[act], np.full(len(act), d / 2 / z))
        charged = np.maximum(charged, n)
        stages.append(dict(radius=d / 2, active=len(act), stage_cost=cost))
    return charged.sum(), charged, stages


def per_gradient_groups(A, xs, zs, nq):
    A = sp.csr_matrix(A)
    out = []
    for i in range(A.shape[0]):
        sup = A.indices[A.indptr[i]:A.indptr[i + 1]]
        c = A.data[A.indptr[i]:A.indptr[i + 1]]
        o = np.argsort(sup); sup, c = sup[o], c[o]
        gid, ng = group_fc(xs[sup], zs[sup], np.argsort(-np.round(np.abs(c), 10), kind="stable"), nq)
        out.append([sup[m] for m in split(gid, ng)])
    return out


def shared_fraction(A, active):
    cnt = np.asarray((abs(A[active]) > 0).sum(0)).ravel()
    union = int((cnt > 0).sum())
    return (cnt >= 2).sum() / max(union, 1), union


def regroup(A, xs, zs, active, nq):
    sub = A[active]
    sup = np.flatnonzero(np.asarray((abs(sub) > 0).sum(0)).ravel() > 0)
    w = np.round(np.sqrt(np.asarray(sub[:, sup].multiply(sub[:, sup]).sum(0)).ravel()), 10)
    gid, ng = group_fc(xs[sup], zs[sup], np.argsort(-w, kind="stable"), nq)
    return [sup[m] for m in split(gid, ng)]


def m3r_cost(state, xs, zs, A, q, g, z, nq, threshold=0.0, delta=0.0):
    A = sp.csr_matrix(A)
    w, gap, D = gaps(g, delta)
    levels = np.unique(np.round(D, 12))[::-1]
    parent = np.zeros(q.shape[1])
    fresh, qn, groups_new, trigger, v_old, rows = None, None, None, None, None, None
    stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (q.sum(1) > 0))
        if len(act) == 0:
            continue
        frac, union = shared_fraction(A, act)
        if groups_new is None and frac <= threshold:
            groups_new = regroup(A, xs, zs, act, nq)
            qn, _ = fragment_variances(state, A[act], xs, zs, groups_new)
            rows = {i: r for r, i in enumerate(act)}
            fresh = np.zeros(len(groups_new))
            v_old = {}
            for i in act:
                used = q[i] > 0
                v_old[i] = np.sum(q[i][used] / parent[used]) if np.all(parent[used] > 0) else np.inf
            trigger = dict(radius=d / 2, active=len(act), union_terms=union, new_groups=len(groups_new))
        eps = np.full(len(act), d / 2 / z)
        if groups_new is None:
            cost, n = allocate(q[act], eps)
            parent = np.maximum(parent, n)
        else:
            need = []
            for i, e in zip(act, eps):
                inv = 1.0 / e ** 2 - (1.0 / v_old[i] if np.isfinite(v_old[i]) else 0.0)
                need.append(1.0 / inv if inv > 0 else np.inf)
            need = np.array(need)
            keep = np.isfinite(need)
            if keep.any():
                cost, n = allocate(qn[[rows[i] for i in act[keep]]], np.sqrt(need[keep]))
                fresh = np.maximum(fresh, n)
            else:
                cost = 0.0
        stages.append(dict(radius=d / 2, active=len(act), shared_fraction=float(frac),
                           regrouped=groups_new is not None, stage_cost=cost))
    total = parent.sum() + (fresh.sum() if fresh is not None else 0.0)
    return total, stages, trigger, groups_new


def fragmentation(A, groups, i):
    A = sp.csr_matrix(A)
    gid = np.empty(A.shape[1], dtype=np.int64)
    for a, mem in enumerate(groups):
        gid[mem] = a
    return len(set(gid[A.indices[A.indptr[i]:A.indptr[i + 1]]]))


Overwriting meas.py


### `circuits.py` — two-qubit gate counts

In [6]:
%%writefile circuits.py
"""Clifford diagonalization gate counts (Part I notebook routine, gate list only)."""
import numpy as np


def gf2_basis(xs, zs, nq):
    rows, piv = [], []
    for x, z in zip(xs, zs):
        v = (int(x) << nq) | int(z)
        for p, r in zip(piv, rows):
            if v >> p & 1:
                v ^= r
        if v:
            p = v.bit_length() - 1
            for k in range(len(rows)):
                if rows[k] >> p & 1:
                    rows[k] ^= v
            rows.append(v); piv.append(p)
    X = np.array([[(r >> (2 * nq - 1 - q)) & 1 for q in range(nq)] for r in rows], dtype=np.uint8).reshape(-1, nq)
    Z = np.array([[(r >> (nq - 1 - q)) & 1 for q in range(nq)] for r in rows], dtype=np.uint8).reshape(-1, nq)
    return X, Z


def two_qubit_count(xs, zs, nq):
    X, Z = gf2_basis(xs, zs, nq)
    k = X.shape[0]
    n2 = 0

    def h(q):
        X[:, q], Z[:, q] = Z[:, q].copy(), X[:, q].copy()

    def rref():
        pivots, r = [], 0
        for q in range(nq):
            rows = np.flatnonzero(X[r:, q]) + r
            if len(rows) == 0:
                continue
            p = rows[0]
            if p != r:
                X[[r, p]] = X[[p, r]]; Z[[r, p]] = Z[[p, r]]
            for rr in np.flatnonzero(X[:, q]):
                if rr != r:
                    X[rr] ^= X[r]; Z[rr] ^= Z[r]
            pivots.append(q); r += 1
            if r == k:
                break
        return pivots

    while True:
        pivots = rref()
        if len(pivots) == k:
            break
        r = len(pivots)
        free = [q for q in range(nq) if Z[r, q] and q not in pivots]
        h(free[0])
    for r, p in enumerate(pivots):
        for q in np.flatnonzero(X[r]):
            if q != p:
                n2 += 1; X[:, q] ^= X[:, p]; Z[:, p] ^= Z[:, q]
    for r, p in enumerate(pivots):
        if Z[r, p]:
            Z[:, p] ^= X[:, p]
    for a in range(k):
        for b in range(a + 1, k):
            if Z[a, pivots[b]]:
                n2 += 1; Z[:, pivots[a]] ^= X[:, pivots[b]]; Z[:, pivots[b]] ^= X[:, pivots[a]]
    return n2


Overwriting circuits.py


### `run_v2.py` — runner: one window geometry, four states, all methods

In [7]:
%%writefile run_v2.py
"""Unified runner: one case geometry, all states, Part I methods + Delta-good + M3-C + M3-R(Delta).
Usage: python run_v2.py H2O_ccpvdz_8o 0 sym [HF,CISD,CASCI,CASSCF]"""
import os, sys, json, time
import numpy as np, scipy.sparse as sp
from windows2 import CASES, build_case, window_mf, choose_cas, cas_states, overlaps, ARCHIVE_CAS, CAS_SIZE
from paulis import hamiltonian, uccsd_pool, gradient_matrix
from meas import *
from states import hf_state, cisd_vector, to_jw, energy
from circuits import two_qubit_count

key, k, POOL = sys.argv[1], int(sys.argv[2]), sys.argv[3]
WANT = sys.argv[4].split(",") if len(sys.argv) > 4 else ["HF", "CISD", "CASCI", "CASSCF"]
DELTA = 1e-3
TRUNC = 1e-6            # discarded weight when a state is truncated for measurement
PAIR_BUDGET = 1.5e9     # (Pauli terms) x (determinants) allowed for one state
OUT = "results/v2"; os.makedirs(OUT, exist_ok=True)
t0 = time.time()
log = lambda *a: print(f"[{time.time()-t0:7.1f}s]", *a, flush=True)

W = build_case(key)[k]
_prev_rows = []
system = W["system"]
tag0 = f"{key}_{W['R']:.4g}_{POOL}"
_sf = f"{OUT}/{key}_{W['R']:.4g}_{POOL}_summary.json"
if os.path.exists(_sf):                      # resume: keep finished states, run only the rest
    _prev_rows = json.load(open(_sf))
    WANT = [s_ for s_ in WANT if s_ not in {r["state"] for r in _prev_rows}]
    if not WANT:
        print("nothing to do"); sys.exit(0)
info = dict(key=key, system=system, basis=W["basis"], window=W["window"], R=W["R"], pool_type=POOL,
            norb=W["ncas"], nelec=W["nelec"], win_irreps=W["win_irreps"], e_hf=W["e_hf"], e_fci=W["e_fci"],
            ref_hf=W["ref_hf"], ref_fci=W["ref_fci"])
log(tag0, "window", W["ncas"], "orbitals; E_HF", W["e_hf"], "E_FCI(win)", W["e_fci"])

# ---------------- Hamiltonian, pool, gradient operators
nq, Hx, Hz, Hc = hamiltonian(W); ne = W["nelec"]
pool, labels = uccsd_pool(nq, ne)
if POOL == "sym":
    mol = W["mol"]
    ids = [mol.irrep_id[mol.irrep_name.index(n)] for n in W["win_irreps"]]
    def sym_ok(lab):
        a, b = lab.split(" ", 1)[1].split("->")
        x = 0
        for o in a.split(",") + b.split(","):
            x ^= ids[int(o) // 2]
        return x == 0
    keep = [i for i, l in enumerate(labels) if sym_ok(l)]
    pool = [pool[i] for i in keep]; labels = [labels[i] for i in keep]
A, xs, zs = gradient_matrix(nq, Hx, Hz, Hc, pool); A = A.tocsr()
log("nq", nq, "H terms", len(Hx), "pool", len(pool), "L", A.shape[1], "nnz", A.nnz)
info.update(qubits=nq, H_terms=len(Hx), pool=len(pool), universal_terms=A.shape[1])

# ---------------- states
psi_all, sinfo = {}, {}
if "HF" in WANT:
    psi_all["HF"] = hf_state(nq, ne); sinfo["HF"] = dict(E=W["e_hf"], ndet=1, kept_weight=1.0)
if "CISD" in WANT:
    civ, e_cisd, _ = cisd_vector(W)
    psi_all["CISD"] = to_jw(civ, W["ncas"], ne); sinfo["CISD"] = dict(E=e_cisd, ndet=len(psi_all["CISD"].idx), kept_weight=1.0)
if "CASCI" in WANT or "CASSCF" in WANT:
    wmf = window_mf(W)
    if (system, round(W["R"], 4)) in ARCHIVE_CAS and W["basis"] == "cc-pvdz" and W["window"] in ("8o", "11o") and system == ("H2O" if W["window"] == "8o" else "N2"):
        cas, _ = choose_cas(W, wmf)
    elif system == "H2O":   # chemical rule, identical to the archive choice where that is known
        irr = W["win_irreps"]; no = ne // 2
        from windows2 import h2o_inplane_b
        b_in = h2o_inplane_b(W["mol"])
        occ_a1 = [i for i in range(no) if irr[i] == "A1"][-1]
        occ_b = [i for i in range(no) if irr[i] == b_in][-1]
        vir_a1 = [i for i in range(no, W["ncas"]) if irr[i] == "A1"][0]
        vir_b = [i for i in range(no, W["ncas"]) if irr[i] == b_in][0]
        cas = sorted([occ_a1, occ_b, vir_a1, vir_b])
    else:                   # by overlap with the validated cc-pVDZ 11o CAS at the same geometry
        R0 = [w for w in build_case("N2_ccpvdz_11o") if abs(w["R"] - W["R"]) < 1e-9][0]
        c0, _ = choose_cas(R0, window_mf(R0))
        rC = R0["mf"].mo_coeff[:, [R0["win"][c] for c in c0]]
        cas, _ = choose_cas(W, wmf, (R0["mol"], rC))
    cs = cas_states(W, cas)
    info["cas_orbitals"] = [W["win_irreps"][c] for c in cas]
    info["E_CASCI"], info["E_CASSCF"] = cs["CASCI"]["e_mc"], cs["CASSCF"]["e_mc"]
    info["ovl_HF_FCI"], info["ovl_CASSCF_FCI"], info["w_FCI_in_CAS"] = overlaps(W, cs)
    for kind in ("CASCI", "CASSCF"):
        if kind not in WANT:
            continue
        st = to_jw(cs[kind]["vec"], W["ncas"], ne)
        a2 = np.abs(st.amp) ** 2; o = np.argsort(-a2); c = np.cumsum(a2[o])
        n = int(np.searchsorted(c, 1 - TRUNC)) + 1
        n = min(n, max(50, int(PAIR_BUDGET / A.shape[1])))
        kept = float(c[n - 1])
        full_ndet = len(st.idx)
        from meas import State
        st = State(st.idx[o[:n]], st.amp[o[:n]])
        psi_all[kind] = st
        sinfo[kind] = dict(E=cs[kind]["e_mc"], ndet=n, ndet_full=full_ndet, kept_weight=kept)
    log("CAS", info["cas_orbitals"], "E_CASCI", info["E_CASCI"], "E_CASSCF", info["E_CASSCF"])

# ---------------- groups
groups = base_partition(A, xs, zs, nq); log("parent groups", len(groups))
n2q = np.array([two_qubit_count(xs[m], zs[m], nq) for m in groups])
pg = per_gradient_groups(A, xs, zs, nq); log("per-gradient groups", sum(len(p) for p in pg))
info.update(contexts=len(groups), M2_groups=int(sum(len(p) for p in pg)), mean_2q=float(n2q.mean()))
json.dump(dict(generators=labels, groups=[m.tolist() for m in groups], n_2q=n2q.tolist()), open(f"{OUT}/{tag0}_groups.json", "w"))
sp.save_npz(f"{OUT}/{tag0}_A.npz", A)
np.savez_compressed(f"{OUT}/{tag0}_paulis.npz", xs=xs, zs=zs, nq=nq)


def m3c_cost(qc, D, w, z):
    levels = np.unique(np.round(D, 12))[::-1]
    charged = np.zeros(qc.shape[1])
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (np.arange(len(D)) != w) & (qc.sum(1) > 0))
        if len(act) == 0:
            continue
        cost, n = allocate(qc[act], np.full(len(act), d / z))
        charged = np.maximum(charged, n)
    return charged.sum()


rows = list(_prev_rows)
for st, psi in psi_all.items():
    e_trunc = energy(psi, Hx, Hz, Hc)
    g = A @ pauli_expect(psi, xs, zs)
    z = z_value(len(g))
    q, mu = fragment_variances(psi, A, xs, zs, groups)
    assert np.abs(mu.sum(1) - g).max() < 1e-8
    ql = [fragment_variances(psi, A[i], xs, zs, pg[i])[0][0] for i in range(len(g))]
    s = np.sign(g); s[s == 0] = 1; w = int(np.argmax(np.abs(g)))
    Cm = (sp.csr_matrix(np.full((A.shape[0], 1), s[w])) @ A[w] - sp.diags(s) @ A).tocsr()
    qc, _ = fragment_variances(psi, Cm, xs, zs, groups)
    res = {}
    for name, dl in (("", 0.0), ("_Delta", DELTA)):
        res["M1" + name] = m1_cost(q, g, z, dl)[0]
        res["M2" + name] = m2_cost(ql, g, z, dl)[0]
        res["M3" + name], _, stages = m3_cost(q, g, z, dl)
        c3r, _, trig, _ = m3r_cost(psi, xs, zs, A, q, g, z, nq, delta=dl)
        res["M3R" + name] = c3r
        res["M3C" + name] = m3c_cost(qc, gaps(g, dl)[2], w, z)
        if name == "":
            res["regroup_active"] = trig["active"] if trig else 0
            res["final_stage_share_M3"] = float(stages[-1]["stage_cost"] / res["M3"])
    _, gap, _ = gaps(g); order = np.argsort(-np.abs(g))
    row = dict(info, state=st, **sinfo[st], E_measured=e_trunc, E_state_minus_FCI_mHa=1e3 * (sinfo[st]["E"] - W["e_fci"]),
               nonzero_g=int(np.sum(np.abs(g) > 1e-8)), top=labels[order[0]], second=labels[order[1]],
               g1=float(abs(g[order[0]])), g2=float(abs(g[order[1]])), gap=float(gap), z=float(z),
               n_Delta_good=int(np.sum(np.abs(g) >= np.abs(g).max() - DELTA)),
               winner_parent_contexts=fragmentation(A, groups, order[0]), winner_own_groups=len(pg[order[0]]),
               **res, seconds=time.time() - t0)
    rows.append(row)
    np.savez_compressed(f"{OUT}/{tag0}_{st}_fragments.npz", q=q, qc=qc, g=g)
    json.dump(dict(gradients=g.tolist(), labels=labels), open(f"{OUT}/{tag0}_{st}_gradients.json", "w"))
    log(st, f"E-EFCI={row['E_state_minus_FCI_mHa']:.2f} mHa ndet={row['ndet']} top={row['top']} g1={row['g1']:.6f} g2={row['g2']:.6f} "
        f"| D: M1={res['M1_Delta']:.3g} M2={res['M2_Delta']:.3g} M3={res['M3_Delta']:.3g} M3R={res['M3R_Delta']:.3g} M3C={res['M3C_Delta']:.3g}")
    json.dump(rows, open(f"{OUT}/{tag0}_summary.json", "w"), indent=1, default=float)
log("done")


Overwriting run_v2.py


### `tables2.py` — result tables

In [8]:
%%writefile tables2.py
"""Tables from results/v2 (all cases, all states)."""
import json, glob
import numpy as np
import pandas as pd

IN = "results/v2"
CASE_ORDER = ["H2O_ccpvdz_8o", "H2O_ccpvdz_11o", "H2O_ccpvdz_14o", "H2O_ccpvtz_8o", "H2O_631g_all",
              "N2_ccpvdz_11o", "N2_ccpvtz_11o"]
STATE_ORDER = ["HF", "CISD", "CASCI", "CASSCF"]
LABEL = {"H2O_ccpvdz_8o": "H2O cc-pVDZ 8o", "H2O_ccpvdz_11o": "H2O cc-pVDZ 11o", "H2O_ccpvdz_14o": "H2O cc-pVDZ 14o",
         "H2O_ccpvtz_8o": "H2O cc-pVTZ 8o", "H2O_631g_all": "H2O 6-31G (all)", "N2_ccpvdz_11o": "N2 cc-pVDZ 11o",
         "N2_ccpvtz_11o": "N2 cc-pVTZ 11o"}


def load():
    rows = []
    for f in sorted(glob.glob(f"{IN}/*_summary.json")):
        rows += json.load(open(f))
    df = pd.DataFrame(rows)
    df["ci"] = df.key.map(CASE_ORDER.index)
    df["si"] = df.state.map(STATE_ORDER.index)
    df["case"] = [f"{LABEL[k]} {r:g} Å" for k, r in zip(df.key, df.R)]
    return df.sort_values(["pool_type", "ci", "R", "si"], ascending=[False, True, True, True]).reset_index(drop=True)


def sci(x):
    return f"{x:.2e}" if np.isfinite(x) else "–"


def validation(df):
    d = df.drop_duplicates(["key", "R"])
    arch = {("H2O", 0.9584): (-76.030123418, -76.054064392, 0.982338, 0.997319, 0.997383),
            ("H2O", 1.75): (-75.799299906, -75.824182389, 0.748394, 0.997991, 0.998177),
            ("N2", 1.1): (-109.021904308, -109.023889473, 0.933468, 0.982545, 0.982729),
            ("N2", 1.8): (-108.757342076, -108.777835125, 0.000232, 0.979468, 0.982738),
            ("N2", 3.0): (-108.695809945, -108.738528934, 1.969e-08, 0.994805, 0.994961)}
    out = []
    for _, r in d.iterrows():
        a = arch.get((r.system, round(r.R, 4))) if r.key in ("H2O_ccpvdz_8o", "N2_ccpvdz_11o") else None
        f = lambda x, y: round(1e6 * (x - y), 2) if (y is not None and x is not None and np.isfinite(x)) else None
        out.append(dict(Case=r.case, Qubits=r.qubits, E_RHF=round(r.e_hf, 9), E_FCI_window=round(r.e_fci, 9),
                        dRHF_uHa=f(r.e_hf, r.ref_hf), dFCI_uHa=f(r.e_fci, r.ref_fci),
                        dCASCI_uHa=f(r.get("E_CASCI"), a[0] if a else None), dCASSCF_uHa=f(r.get("E_CASSCF"), a[1] if a else None),
                        HF_FCI=f"{r.get('ovl_HF_FCI', np.nan):.4g}" + (f" ({a[2]:.4g})" if a else ""),
                        CASSCF_FCI=f"{r.get('ovl_CASSCF_FCI', np.nan):.6f}" + (f" ({a[3]:.6f})" if a else ""),
                        W_CAS=f"{r.get('w_FCI_in_CAS', np.nan):.6f}" + (f" ({a[4]:.6f})" if a else "")))
    return pd.DataFrame(out)


def structure(df):
    d = df.drop_duplicates(["key", "R", "pool_type"])
    return pd.DataFrame(dict(Case=d.case, Pool=d.pool_type, Qubits=d.qubits, Electrons=d.nelec, H_terms=d.H_terms,
                             Generators=d.pool, Pauli_terms=d.universal_terms, FC_contexts=d.contexts,
                             M2_groups=d.M2_groups, mean_2q=d.mean_2q.round(1))).reset_index(drop=True)


def gradients(df):
    return pd.DataFrame(dict(Case=df.case, Pool=df.pool_type, State=df.state, E_minus_FCI_mHa=df.E_state_minus_FCI_mHa.round(2),
                             ndet=df.ndet, Top=df.top, g1=df.g1.round(6), g2=df.g2.round(6),
                             gap=df.gap.map(lambda x: f"{x:.1e}"), N_Delta=df.n_Delta_good))


METHODS = ["M1", "M2", "M3", "M3R", "M3C"]


def costs(df, delta=True):
    s = "_Delta" if delta else ""
    out = pd.DataFrame(dict(Case=df.case, Pool=df.pool_type, State=df.state))
    for m in METHODS:
        out[m] = df[m + s].map(sci)
    vals = df[[m + s for m in METHODS]].values
    best = np.argmin(vals, axis=1)
    out["Best"] = [METHODS[b].replace("M3R", "M3-R").replace("M3C", "M3-C") for b in best]
    out["M2/best"] = (df["M2" + s] / vals.min(1)).round(2)
    out["M1/best"] = (df["M1" + s] / vals.min(1)).round(1)
    out["M3/M3C"] = (df["M3" + s] / df["M3C" + s]).round(2)
    return out


Overwriting tables2.py


In [9]:
import os, json, re
os.environ["OMP_NUM_THREADS"] = "2"
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 300)
RUN = False

## 1. Purpose, geometries, and conventions
H₂O: R(OH) = 0.9584, 1.75, 3.00 Å, angle 104.45°, C2v labels. N₂: R = 1.10, 1.80, 3.00 Å, D2h labels.
PySCF (spherical basis, symmetry on); OpenFermion Jordan–Wigner. Retained orbitals are traced between geometries by irrep
and overlap. The window Hamiltonian is the CASCI effective Hamiltonian of the retained orbitals.

## 2. Basis-size comparison and RHF energies
The archive spectrum tables (Tables 5–18) are parsed from the archive PDF text into `archive_check/spectra_full.json`.
RHF is run at every archive geometry; for stretched geometries several initial guesses are tried and the solution closest to
the archive energy is kept.

In [10]:
ARCH = json.load(open("archive_check/spectra_full.json"))
SPEC = [("H2O","6-31g",0.9584,-75.983953320),("H2O","6-31g",1.75,-75.653868837),("H2O","6-31g",3.0,-75.316037599),
        ("N2","6-31g",1.10,-108.867618379),("N2","6-31g",1.80,-108.323966181),("N2","6-31g",3.00,-108.219724168),
        ("H2O","cc-pvdz",0.9584,-76.026741428),("H2O","cc-pvdz",1.75,-75.669624909),("H2O","cc-pvdz",3.0,-75.430561431),
        ("H2O","cc-pvtz",0.95784,-76.051495438),
        ("N2","cc-pvdz",1.10,-108.953796253),("N2","cc-pvdz",1.80,-108.341640217),("N2","cc-pvdz",3.00,-108.231992299),
        ("N2","cc-pvtz",1.09768,-108.983965064)]
TABNO = [5,6,7,8,9,10,11,12,13,17,14,15,16,18]
if RUN:
    from pyscf import gto, scf
    from windows import water, n2, labels
    runs = []
    for sy, bas, r, eref in SPEC:
        mol = gto.M(atom=water(r) if sy=="H2O" else n2(r), basis=bas, symmetry=True if sy=="H2O" else "D2h", verbose=0)
        best = None
        for g, nt in (("minao",False),("1e",True),("huckel",False),("atom",False),("vsap",False)):
            mf = scf.RHF(mol); mf.conv_tol=1e-10; mf.max_cycle=300; mf.init_guess=g
            if nt: mf = mf.newton()
            mf.kernel()
            if best is None or abs(mf.e_tot-eref) < abs(best.e_tot-eref): best = mf
        names = dict(zip(mol.irrep_id, mol.irrep_name))
        runs.append(dict(sy=sy, bas=bas, r=r, eref=eref, e=best.e_tot, eps=best.mo_energy.tolist(),
                         occ=best.mo_occ.tolist(), irr=[names[s] for s in labels(mol, best)]))
    json.dump(runs, open("archive_check/spec_runs.json","w"))
RUNS = json.load(open("archive_check/spec_runs.json"))

### Archive Table 1 — orbital dimensions (archive vs ours)

In [11]:
dims = {("H2O","6-31g"):13,("H2O","cc-pvdz"):24,("H2O","cc-pvtz"):58,("N2","6-31g"):18,("N2","cc-pvdz"):28,("N2","cc-pvtz"):60}
ours = {(r["sy"], r["bas"]): len(r["eps"]) for r in RUNS}
pd.DataFrame([dict(System=s, Basis=b, archive_spatial=d, ours_spatial=ours[(s,b)], archive_spin=2*d, ours_spin=2*ours[(s,b)])
              for (s,b), d in dims.items()])

,System,Basis,archive_spatial,ours_spatial,archive_spin,ours_spin
0,H2O,6-31g,13,13,26,26
1,H2O,cc-pvdz,24,24,48,48
2,H2O,cc-pvtz,58,58,116,116
3,N2,6-31g,18,18,36,36
4,N2,cc-pvdz,28,28,56,56
5,N2,cc-pvtz,60,60,120,120


### Archive Table 2 — RHF energies (and the cc-pVTZ preflight values)

In [12]:
pd.DataFrame([dict(System=r["sy"], Basis=r["bas"], R=r["r"], archive=r["eref"], ours=round(r["e"], 9),
                   diff_uHa=round(1e6*(r["e"]-r["eref"]), 2), match=abs(r["e"]-r["eref"]) < 1e-6) for r in RUNS])

,System,Basis,R,archive,ours,diff_uHa,match
0,H2O,6-31g,0.95840,-75.983953,-75.983953,-0.01,True
1,H2O,6-31g,1.75000,-75.653869,-75.653869,0.05,True
2,H2O,6-31g,3.00000,-75.316038,-75.316038,0.05,True
3,N2,6-31g,1.10000,-108.867618,-108.867618,0.01,True
4,N2,6-31g,1.80000,-108.323966,-108.323966,0.01,True
5,N2,6-31g,3.00000,-108.219724,-108.219724,-0.00,True
6,H2O,cc-pvdz,0.95840,-76.026741,-76.026741,0.00,True
7,H2O,cc-pvdz,1.75000,-75.669625,-75.669625,0.05,True
8,H2O,cc-pvdz,3.00000,-75.430561,-75.435687,-5125.75,False
9,H2O,cc-pvtz,0.95784,-76.051495,-76.057122,-5626.07,False


**Reading:** 11 of the 12 Table 2 energies agree to 0.06 μHa. H₂O cc-pVDZ at 3.0 Å does not: the closest solution
found is 5.1 mHa lower. The cc-pVTZ preflight values were computed at geometries the archive does not fully give, so they are
guidance only.

## 3. Retained-orbital windows
For each window and geometry: every retained orbital with its RHF energy, irrep and role.

In [13]:
from windows2 import build_case, h2o_inplane_b
from windows import labels
def window_table(key):
    out = []
    for W in build_case(key, fci_solve=False):
        mol, mf = W["mol"], W["mf"]
        names = dict(zip(mol.irrep_id, mol.irrep_name)); irr = [names[s] for s in labels(mol, mf)]
        cnt = {}; lab = []
        for x in irr:
            cnt[x] = cnt.get(x, 0) + 1; lab.append(f"{cnt[x]}{x.lower()}")
        nocc = mol.nelectron // 2; win = W["win"]
        occw = sorted(i for i in win if i < nocc); virw = sorted(i for i in win if i >= nocc)
        if W["system"] == "H2O":
            b = h2o_inplane_b(mol)
            cas = [max(i for i in occw if irr[i]=="A1"), max(i for i in occw if irr[i]==b),
                   min(i for i in virw if irr[i]=="A1"), min(i for i in virw if irr[i]==b)]
        else:
            cas = occw[2:] + virw[:3]
        for i in list(W["core"]) + win:
            role = "frozen core" if i in W["core"] else "CAS" if i in cas else "inactive" if i < nocc else "external"
            out.append(dict(R=W["R"], index=i+1, occ=2 if i < nocc else 0, eps=round(mf.mo_energy[i], 6),
                            irrep=irr[i], label=lab[i], role=role))
    return pd.DataFrame(out)

### 3.1 H₂O cc-pVDZ 8o

In [14]:
window_table("H2O_ccpvdz_8o")

,R,index,occ,eps,irrep,label,role
0,0.9584,1,2,-20.550665,A1,1a1,frozen core
1,0.9584,2,2,-1.336155,A1,2a1,inactive
2,0.9584,3,2,-0.698560,B2,1b2,CAS
3,0.9584,4,2,-0.566489,A1,3a1,CAS
4,0.9584,5,2,-0.493087,B1,1b1,inactive
5,0.9584,6,0,0.185363,A1,4a1,CAS
6,0.9584,7,0,0.256091,B2,2b2,CAS
7,0.9584,8,0,0.788274,B2,3b2,external
8,0.9584,9,0,0.853532,A1,5a1,external
9,1.7500,1,2,-20.659582,A1,1a1,frozen core


### 3.1 H₂O cc-pVDZ 11o

In [15]:
window_table("H2O_ccpvdz_11o")

,R,index,occ,eps,irrep,label,role
0,0.9584,1,2,-20.550665,A1,1a1,frozen core
1,0.9584,2,2,-1.336155,A1,2a1,inactive
2,0.9584,3,2,-0.698560,B2,1b2,CAS
3,0.9584,4,2,-0.566489,A1,3a1,CAS
4,0.9584,5,2,-0.493087,B1,1b1,inactive
5,0.9584,6,0,0.185363,A1,4a1,CAS
6,0.9584,7,0,0.256091,B2,2b2,CAS
7,0.9584,8,0,0.788274,B2,3b2,external
8,0.9584,9,0,0.853532,A1,5a1,external
9,0.9584,10,0,1.163636,A1,6a1,external


### 3.2 N₂ cc-pVDZ 11o

In [16]:
window_table("N2_ccpvdz_11o")

,R,index,occ,eps,irrep,label,role
0,1.1,1,2,-15.687167,Ag,1ag,frozen core
1,1.1,2,2,-15.683810,B1u,1b1u,frozen core
2,1.1,3,2,-1.469497,Ag,2ag,inactive
3,1.1,4,2,-0.774851,B1u,2b1u,inactive
4,1.1,5,2,-0.626027,Ag,3ag,CAS
5,1.1,6,2,-0.607175,B3u,1b3u,CAS
6,1.1,7,2,-0.607175,B2u,1b2u,CAS
7,1.1,8,0,0.174428,B2g,1b2g,CAS
8,1.1,9,0,0.174428,B3g,1b3g,CAS
9,1.1,10,0,0.593893,B1u,3b1u,CAS


### H₂O cc-pVTZ 8o (character-defined, 1.75 Å)

In [17]:
window_table("H2O_ccpvtz_8o")

,R,index,occ,eps,irrep,label,role
0,1.75,1,2,-20.649024,A1,1a1,frozen core
1,1.75,2,2,-1.202030,A1,2a1,inactive
2,1.75,3,2,-0.506024,B1,1b1,inactive
3,1.75,4,2,-0.460784,B2,1b2,CAS
4,1.75,5,2,-0.446049,A1,3a1,CAS
5,1.75,6,0,-0.006890,A1,4a1,CAS
6,1.75,7,0,0.033031,B2,2b2,CAS
7,1.75,8,0,0.378926,A1,5a1,external
8,1.75,9,0,0.387767,B2,3b2,external


### N₂ cc-pVTZ 11o (character-defined, 1.80 Å)

In [18]:
window_table("N2_ccpvtz_11o")

,R,index,occ,eps,irrep,label,role
0,1.8,1,2,-15.743796,Ag,1ag,frozen core
1,1.8,2,2,-15.743720,B1u,1b1u,frozen core
2,1.8,3,2,-1.108764,Ag,2ag,inactive
3,1.8,4,2,-0.921464,B1u,2b1u,inactive
4,1.8,5,2,-0.548487,B3u,1b3u,CAS
5,1.8,6,2,-0.451391,B2g,1b2g,CAS
6,1.8,7,2,-0.417706,B2u,1b2u,CAS
7,1.8,8,0,-0.102915,Ag,3ag,CAS
8,1.8,9,0,-0.044104,B3g,1b3g,CAS
9,1.8,10,0,0.188923,B1u,3b1u,CAS


### 3.3 Orbital continuation
Core frozen; every occupied orbital kept; each irrep filled back to its equilibrium count with the virtuals that overlap most
with the previous window. The check below shows why: at N₂ 1.80 Å, taking the lowest virtuals per irrep gives a window FCI
1.06 mHa away from the archive, while overlap tracing matches it.

## 4. Validated windowed HF/CASSCF/FCI diagnostics

In [19]:
from tables2 import load, validation, structure, gradients, costs, LABEL, METHODS
df = load()
ARCH3 = {("H2O",0.9584):(-76.026741428,-76.030123418,-76.054064392,-76.059281484),("H2O",1.75):(-75.669624909,-75.799299906,-75.824182389,-75.826835961),
         ("N2",1.1):(-108.953796253,-109.021904308,-109.023889473,-109.059444517),("N2",1.8):(-108.341640217,-108.757342076,-108.777835125,-108.806606966),
         ("N2",3.0):(-108.231992299,-108.695809945,-108.738528934,-108.749118387)}
ARCH4 = {("H2O",0.9584):(0.982338,0.997319,0.997383),("H2O",1.75):(0.748394,0.997991,0.998177),("N2",1.1):(0.933468,0.982545,0.982729),
         ("N2",1.8):(0.000232,0.979468,0.982738),("N2",3.0):(1.969e-08,0.994805,0.994961)}
val = df[df.key.isin(["H2O_ccpvdz_8o","N2_ccpvdz_11o"]) & (df.pool_type=="sym")].drop_duplicates(["key","R"])
t3 = []
for _, r in val.iterrows():
    a = ARCH3[(r.system, round(r.R,4))]
    t3.append(dict(System=r.system, R=r.R, source="archive", E_RHF=a[0], E_CASCI=a[1], E_CASSCF=a[2], E_FCI=a[3]))
    t3.append(dict(System=r.system, R=r.R, source="ours", E_RHF=r.e_hf, E_CASCI=r.E_CASCI, E_CASSCF=r.E_CASSCF, E_FCI=r.e_fci))
print("Archive Table 3 — energies (Eh)"); pd.DataFrame(t3).round(9)

Archive Table 3 — energies (Eh)


,System,R,source,E_RHF,E_CASCI,E_CASSCF,E_FCI
0,H2O,0.9584,archive,-76.026741,-76.030123,-76.054064,-76.059281
1,H2O,0.9584,ours,-76.026741,-76.030123,-76.054064,-76.059281
2,H2O,1.7500,archive,-75.669625,-75.799300,-75.824182,-75.826836
3,H2O,1.7500,ours,-75.669625,-75.799300,-75.824182,-75.826836
4,N2,1.1000,archive,-108.953796,-109.021904,-109.023889,-109.059445
5,N2,1.1000,ours,-108.953796,-109.021905,-109.023890,-109.059445
6,N2,1.8000,archive,-108.341640,-108.757342,-108.777835,-108.806607
7,N2,1.8000,ours,-108.341640,-108.757342,-108.777835,-108.806607
8,N2,3.0000,archive,-108.231992,-108.695810,-108.738529,-108.749118
9,N2,3.0000,ours,-108.231992,-108.695735,-108.738529,-108.749118


In [20]:
t4 = []
for _, r in val.iterrows():
    a = ARCH3[(r.system, round(r.R,4))]; o = ARCH4[(r.system, round(r.R,4))]
    t4.append(dict(System=r.system, R=r.R, source="archive", dHF=1e3*(a[0]-a[3]), dCASCI=1e3*(a[1]-a[3]), dCASSCF=1e3*(a[2]-a[3]),
                   HF_FCI=o[0], CASSCF_FCI=o[1], W_CAS=o[2]))
    t4.append(dict(System=r.system, R=r.R, source="ours", dHF=1e3*(r.e_hf-r.e_fci), dCASCI=1e3*(r.E_CASCI-r.e_fci),
                   dCASSCF=1e3*(r.E_CASSCF-r.e_fci), HF_FCI=r.ovl_HF_FCI, CASSCF_FCI=r.ovl_CASSCF_FCI, W_CAS=r.w_FCI_in_CAS))
print("Archive Table 4 — errors (mEh) and squared overlaps"); pd.DataFrame(t4).round(6)

Archive Table 4 — errors (mEh) and squared overlaps


,System,R,source,dHF,dCASCI,dCASSCF,HF_FCI,CASSCF_FCI,W_CAS
0,H2O,0.9584,archive,32.540056,29.158066,5.217092,0.982338,0.997319,0.997383
1,H2O,0.9584,ours,32.540063,29.158071,5.217092,0.982469,0.997319,0.997383
2,H2O,1.7500,archive,157.211052,27.536055,2.653572,0.748394,0.997991,0.998177
3,H2O,1.7500,ours,157.211068,27.536049,2.653571,0.748417,0.997991,0.998177
4,N2,1.1000,archive,105.648264,37.540209,35.555044,0.933468,0.982545,0.982729
5,N2,1.1000,ours,105.648813,37.540062,35.554795,0.933647,0.982546,0.982729
6,N2,1.8000,archive,464.966749,49.264890,28.771841,0.000232,0.979468,0.982738
7,N2,1.8000,ours,464.966730,49.264895,28.771837,0.000337,0.979468,0.982738
8,N2,3.0000,archive,517.126088,53.308442,10.589453,0.000000,0.994805,0.994961
9,N2,3.0000,ours,517.126029,53.383651,10.589393,0.000000,0.994764,0.994961


### Added windows (no archive reference)

In [21]:
add = df[~df.key.isin(["H2O_ccpvdz_8o","N2_ccpvdz_11o"]) & (df.pool_type=="sym")].drop_duplicates(["key","R"])
pd.DataFrame(dict(Case=add.case, qubits=add.qubits, E_RHF=add.e_hf.round(6), E_FCI_win=add.e_fci.round(6),
                  dHF_mHa=(1e3*(add.e_hf-add.e_fci)).round(2), dCASCI_mHa=(1e3*(add.E_CASCI-add.e_fci)).round(2),
                  dCASSCF_mHa=(1e3*(add.E_CASSCF-add.e_fci)).round(2), HF_FCI=add.ovl_HF_FCI.round(4),
                  CASSCF_FCI=add.ovl_CASSCF_FCI.round(6), W_CAS=add.w_FCI_in_CAS.round(6))).reset_index(drop=True)

,Case,qubits,E_RHF,E_FCI_win,dHF_mHa,dCASCI_mHa,dCASSCF_mHa,HF_FCI,CASSCF_FCI,W_CAS
0,H2O cc-pVDZ 11o 0.9584 Å,22,-76.026741,-76.126607,99.87,96.48,58.21,0.9634,0.978727,0.979058
1,H2O cc-pVDZ 11o 1.75 Å,22,-75.669625,-75.882291,212.67,82.99,36.81,0.7364,0.982268,0.983874
2,H2O cc-pVTZ 8o 1.75 Å,16,-75.692858,-75.835674,142.82,32.64,2.69,0.7673,0.997703,0.997864
3,H2O 6-31G (all) 0.9584 Å,24,-75.983953,-76.119998,136.04,128.90,82.92,0.9550,0.973638,0.973895
4,H2O 6-31G (all) 1.75 Å,24,-75.653869,-75.908025,254.16,111.83,72.72,0.7250,0.971771,0.974128
5,H2O 6-31G (all) 3 Å,24,-75.316038,-75.836239,520.20,98.80,60.62,0.2253,0.979547,0.979646
6,N2 cc-pVTZ 11o 1.8 Å,22,-108.367877,-108.794811,426.93,41.30,25.44,0.0003,0.979622,0.982447


## 5. Basis-set and production recommendations
(a) cc-pVDZ main basis — done. (b) 6-31G development benchmark — H₂O, all non-core orbitals, 3 geometries.
(c) cc-pVTZ with a character-defined window — H₂O 1.75 Å and N₂ 1.80 Å. (d) intermediate geometries — included everywhere.

## 6. ADAPT-VQE gradient-measurement costs (new)
Oracle context-shots, 95% confidence, Δ-good target (Δ = 1 mHa). M1: all gradients on shared FC groups. M2: BAI, own groups
per gradient. M3: BAI on shared groups. M3-R: M3 + regrouping. M3-C: M3 with one interval on |g_w| − |g_j|.

In [22]:
if RUN:
    import subprocess
    for k, g, p in [("H2O_ccpvdz_8o",0,"sym"),("H2O_ccpvdz_8o",1,"sym"),("H2O_ccpvdz_8o",0,"full"),("H2O_ccpvdz_8o",1,"full"),
                    ("N2_ccpvdz_11o",0,"sym"),("N2_ccpvdz_11o",1,"sym"),("N2_ccpvdz_11o",2,"sym"),
                    ("H2O_ccpvdz_11o",0,"sym"),("H2O_ccpvdz_11o",1,"sym"),("H2O_ccpvtz_8o",0,"sym"),("H2O_ccpvtz_8o",0,"full"),
                    ("N2_ccpvtz_11o",0,"sym"),("H2O_631g_all",0,"sym"),("H2O_631g_all",1,"sym"),("H2O_631g_all",2,"sym")]:
        subprocess.run(["python3","run_v2.py",k,str(g),p], check=True)
    df = load()

### Problem sizes

In [23]:
structure(df)

,Case,Pool,Qubits,Electrons,H_terms,Generators,Pauli_terms,FC_contexts,M2_groups,mean_2q
0,H2O cc-pVDZ 8o 0.9584 Å,sym,16,8,1905,112,61303,1201,11717,17.6
1,H2O cc-pVDZ 8o 1.75 Å,sym,16,8,1905,112,61303,1188,11843,17.4
2,H2O cc-pVDZ 11o 0.9584 Å,sym,22,8,6078,308,414415,5508,63677,31.2
3,H2O cc-pVDZ 11o 1.75 Å,sym,22,8,6078,308,414415,5498,65985,30.8
4,H2O cc-pVTZ 8o 1.75 Å,sym,16,8,1905,112,61303,1218,11903,17.3
5,H2O 6-31G (all) 0.9584 Å,sym,24,8,8921,408,709332,7905,110220,38.3
6,H2O 6-31G (all) 1.75 Å,sym,24,8,8921,408,709332,7867,111976,37.9
7,H2O 6-31G (all) 3 Å,sym,24,8,8921,408,709332,7795,110866,38.0
8,N2 cc-pVDZ 11o 1.1 Å,sym,22,10,2850,172,206547,3324,19179,31.2
9,N2 cc-pVDZ 11o 1.8 Å,sym,22,10,2850,170,207473,3312,18911,31.3


### Leading gradients (sym pool)

In [24]:
gradients(df[df.pool_type=="sym"])

,Case,Pool,State,E_minus_FCI_mHa,ndet,Top,g1,g2,gap,N_Delta
0,H2O cc-pVDZ 8o 0.9584 Å,sym,HF,32.54,1,"D 2,3->12,13",0.179271,0.146089,3.3e-02,1
1,H2O cc-pVDZ 8o 0.9584 Å,sym,CISD,0.40,113,"D 2,3->12,13",0.002195,0.002189,6.2e-06,11
2,H2O cc-pVDZ 8o 0.9584 Å,sym,CASCI,29.16,19,"D 2,3->12,13",0.176695,0.144439,3.2e-02,1
3,H2O cc-pVDZ 8o 0.9584 Å,sym,CASSCF,5.22,248,"D 4,5->12,13",0.074883,0.049361,2.6e-02,1
4,H2O cc-pVDZ 8o 1.75 Å,sym,HF,157.21,1,"D 4,5->10,11",0.274445,0.245186,2.9e-02,1
5,H2O cc-pVDZ 8o 1.75 Å,sym,CISD,16.49,113,"D 4,5->10,11",0.052233,0.043816,8.4e-03,1
6,H2O cc-pVDZ 8o 1.75 Å,sym,CASCI,27.54,20,"D 4,5->10,15",0.085935,0.085935,8.3e-17,2
7,H2O cc-pVDZ 8o 1.75 Å,sym,CASSCF,2.65,226,"D 0,1->10,11",0.057279,0.053346,3.9e-03,1
8,H2O cc-pVDZ 11o 0.9584 Å,sym,HF,99.87,1,"D 6,7->18,19",0.319433,0.226688,9.3e-02,1
9,H2O cc-pVDZ 11o 0.9584 Å,sym,CISD,3.97,309,"D 6,7->18,19",0.019773,0.012958,6.8e-03,1


### Δ-good costs, sym pool

In [25]:
costs(df[df.pool_type=="sym"], delta=True)

,Case,Pool,State,M1,M2,M3,M3R,M3C,Best,M2/best,M1/best,M3/M3C
0,H2O cc-pVDZ 8o 0.9584 Å,sym,HF,2.60e+07,7.22e+06,4.38e+06,4.38e+06,1.87e+06,M3-C,3.86,13.9,2.35
1,H2O cc-pVDZ 8o 0.9584 Å,sym,CISD,2.89e+10,4.17e+10,1.72e+10,1.72e+10,5.96e+09,M3-C,7.00,4.9,2.89
2,H2O cc-pVDZ 8o 0.9584 Å,sym,CASCI,2.77e+07,7.51e+06,4.67e+06,4.67e+06,2.00e+06,M3-C,3.76,13.8,2.34
3,H2O cc-pVDZ 8o 0.9584 Å,sym,CASSCF,4.44e+07,3.38e+07,1.63e+07,1.63e+07,6.14e+06,M3-C,5.51,7.2,2.66
4,H2O cc-pVDZ 8o 1.75 Å,sym,HF,3.34e+07,1.86e+06,2.25e+06,1.42e+06,1.02e+06,M3-C,1.83,32.8,2.21
5,H2O cc-pVDZ 8o 1.75 Å,sym,CISD,4.10e+08,4.04e+07,3.49e+07,2.18e+07,1.57e+07,M3-C,2.57,26.1,2.23
6,H2O cc-pVDZ 8o 1.75 Å,sym,CASCI,2.89e+10,1.08e+09,2.68e+09,2.68e+09,1.25e+09,M2,1.00,26.7,2.14
7,H2O cc-pVDZ 8o 1.75 Å,sym,CASSCF,1.85e+09,1.28e+08,2.79e+08,2.79e+08,1.33e+08,M2,1.00,14.4,2.09
8,H2O cc-pVDZ 11o 0.9584 Å,sym,HF,1.21e+07,7.50e+06,2.02e+06,1.92e+06,6.90e+05,M3-C,10.87,17.5,2.92
9,H2O cc-pVDZ 11o 0.9584 Å,sym,CISD,2.22e+09,1.95e+09,4.77e+08,4.71e+08,1.59e+08,M3-C,12.28,14.0,3.01


### Exact-winner costs, sym pool

In [26]:
costs(df[df.pool_type=="sym"], delta=False)

,Case,Pool,State,M1,M2,M3,M3R,M3C,Best,M2/best,M1/best,M3/M3C
0,H2O cc-pVDZ 8o 0.9584 Å,sym,HF,2.60e+07,7.22e+06,4.38e+06,4.38e+06,1.87e+06,M3-C,3.86,13.9,2.35
1,H2O cc-pVDZ 8o 0.9584 Å,sym,CISD,7.49e+14,3.17e+13,5.96e+13,5.96e+13,2.57e+13,M3-C,1.23,29.1,2.32
2,H2O cc-pVDZ 8o 0.9584 Å,sym,CASCI,2.77e+07,7.51e+06,4.67e+06,4.67e+06,2.00e+06,M3-C,3.76,13.8,2.34
3,H2O cc-pVDZ 8o 0.9584 Å,sym,CASSCF,4.44e+07,3.38e+07,1.63e+07,1.63e+07,6.14e+06,M3-C,5.51,7.2,2.66
4,H2O cc-pVDZ 8o 1.75 Å,sym,HF,3.34e+07,1.86e+06,2.25e+06,1.42e+06,1.02e+06,M3-C,1.83,32.8,2.21
5,H2O cc-pVDZ 8o 1.75 Å,sym,CISD,4.10e+08,4.04e+07,3.49e+07,2.18e+07,1.57e+07,M3-C,2.57,26.1,2.23
6,H2O cc-pVDZ 8o 1.75 Å,sym,CASCI,4.17e+36,1.47e+35,–,–,–,M2,1.00,28.3,NaN
7,H2O cc-pVDZ 8o 1.75 Å,sym,CASSCF,1.85e+09,1.28e+08,2.79e+08,2.79e+08,1.33e+08,M2,1.00,14.4,2.09
8,H2O cc-pVDZ 11o 0.9584 Å,sym,HF,1.21e+07,7.50e+06,2.02e+06,1.92e+06,6.90e+05,M3-C,10.87,17.5,2.92
9,H2O cc-pVDZ 11o 0.9584 Å,sym,CISD,2.22e+09,1.95e+09,4.77e+08,4.71e+08,1.59e+08,M3-C,12.28,14.0,3.01


### Δ-good costs, full pool (H₂O)

In [27]:
costs(df[df.pool_type=="full"], delta=True)

,Case,Pool,State,M1,M2,M3,M3R,M3C,Best,M2/best,M1/best,M3/M3C
48,H2O cc-pVDZ 8o 0.9584 Å,full,HF,8.12e+07,1.54e+07,8.50e+06,8.50e+06,3.42e+06,M3-C,4.50,23.7,2.48
49,H2O cc-pVDZ 8o 0.9584 Å,full,CISD,8.96e+10,9.66e+10,3.76e+10,3.76e+10,1.27e+10,M3-C,7.58,7.0,2.95
50,H2O cc-pVDZ 8o 0.9584 Å,full,CASCI,8.59e+07,1.60e+07,9.03e+06,9.03e+06,3.65e+06,M3-C,4.39,23.5,2.47
51,H2O cc-pVDZ 8o 0.9584 Å,full,CASSCF,1.38e+08,8.06e+07,3.49e+07,3.49e+07,1.26e+07,M3-C,6.39,10.9,2.76
52,H2O cc-pVDZ 8o 1.75 Å,full,HF,1.25e+08,4.68e+06,4.07e+06,2.83e+06,1.71e+06,M3-C,2.74,73.1,2.38
53,H2O cc-pVDZ 8o 1.75 Å,full,CISD,1.48e+09,1.19e+08,7.33e+07,5.37e+07,2.97e+07,M3-C,4.01,49.8,2.47
54,H2O cc-pVDZ 8o 1.75 Å,full,CASCI,1.05e+11,1.30e+09,4.60e+09,4.60e+09,2.14e+09,M2,1.00,80.9,2.15
55,H2O cc-pVDZ 8o 1.75 Å,full,CASSCF,6.66e+09,2.09e+08,4.15e+08,4.15e+08,1.99e+08,M3-C,1.05,33.5,2.08
56,H2O cc-pVTZ 8o 1.75 Å,full,HF,1.03e+08,4.40e+06,4.06e+06,2.86e+06,1.72e+06,M3-C,2.55,59.7,2.35
57,H2O cc-pVTZ 8o 1.75 Å,full,CISD,1.81e+09,1.37e+08,9.52e+07,6.56e+07,3.97e+07,M3-C,3.45,45.6,2.40


### Summary numbers

In [28]:
s = df[df.pool_type=="sym"]; v = s[[m+"_Delta" for m in METHODS]].values; best = v.min(1)
print("cases:", len(s), "| M3-C or M3-R cheapest:", int((s.M2_Delta.values > best*(1+1e-9)).sum()))
print("M3/M3-C range: %.2f-%.2f" % ((s.M3_Delta/s.M3C_Delta).min(), (s.M3_Delta/s.M3C_Delta).max()))
print("median M2/best per window:"); print((s.M2_Delta/best).groupby(s.key).median().round(1))

cases: 48 | M3-C or M3-R cheapest: 42
M3/M3-C range: 2.01-3.56
median M2/best per window:
key
H2O_631g_all      10.9
H2O_ccpvdz_11o    10.8
H2O_ccpvdz_8o      3.2
H2O_ccpvtz_8o      1.2
N2_ccpvdz_11o      1.9
N2_ccpvtz_11o      6.4
Name: M2_Delta, dtype: float64


## A/B. Archive Tables 5–18: every orbital energy, archive vs ours
Δε in μHa. In N₂ the labels of x/y partners can appear in the other order at equal energy.

In [29]:
def spectrum_table(k):
    a, r = ARCH[k], RUNS[k]
    rows = []
    for i, x in enumerate(a):
        eo = r["eps"][i]
        rows.append(dict(index=x[0], occ=x[1], archive_eps=x[2], ours_eps=round(eo, 6), d_uHa=round(1e6*(eo-x[2]), 2),
                         archive_irrep=x[3], ours_irrep=r["irr"][i], archive_label=x[4]))
    print(f"Archive Table {TABNO[k]}: {r['sy']}/{r['bas']} R={r['r']}  E_RHF archive={r['eref']:.9f} ours={r['e']:.9f}")
    return pd.DataFrame(rows)

In [30]:
spectrum_table(0)

Archive Table 5: H2O/6-31g R=0.9584  E_RHF archive=-75.983953320 ours=-75.983953326


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.560668,-20.560668,-0.21,A1,A1,1a1
1,2,2,-1.355831,-1.355831,-0.23,A1,A1,2a1
2,3,2,-0.709439,-0.709439,0.36,B2,B2,1b2
3,4,2,-0.560578,-0.560578,0.00,A1,A1,3a1
4,5,2,-0.501349,-0.501349,0.45,B1,B1,1b1
5,6,0,0.203488,0.203488,-0.10,A1,A1,4a1
6,7,0,0.299549,0.299549,-0.28,B2,B2,2b2
7,8,0,1.056838,1.056838,0.23,B2,B2,3b2
8,9,0,1.164434,1.164434,-0.18,B1,B1,2b1
9,10,0,1.186123,1.186123,0.16,A1,A1,5a1


In [31]:
spectrum_table(1)

Archive Table 6: H2O/6-31g R=1.75  E_RHF archive=-75.653868837 ours=-75.653868788


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.666400,-20.666401,-0.75,A1,A1,1a1
1,2,2,-1.203060,-1.203060,-0.12,A1,A1,2a1
2,3,2,-0.507061,-0.507061,-0.08,B1,B1,1b1
3,4,2,-0.457241,-0.457241,0.12,B2,B2,1b2
4,5,2,-0.442724,-0.442724,-0.10,A1,A1,3a1
5,6,0,0.003648,0.003648,-0.26,A1,A1,4a1
6,7,0,0.045761,0.045761,-0.36,B2,B2,2b2
7,8,0,0.930148,0.930148,0.25,A1,A1,5a1
8,9,0,1.002403,1.002403,-0.35,B2,B2,3b2
9,10,0,1.157482,1.157482,-0.43,B1,B1,2b1


In [32]:
spectrum_table(2)

Archive Table 7: H2O/6-31g R=3.0  E_RHF archive=-75.316037599 ours=-75.316037552


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.675594,-20.675594,-0.22,A1,A1,1a1
1,2,2,-1.192454,-1.192453,0.53,A1,A1,2a1
2,3,2,-0.514483,-0.514483,0.39,B1,B1,1b1
3,4,2,-0.338331,-0.338331,-0.12,B2,B2,1b2
4,5,2,-0.333586,-0.333585,0.68,A1,A1,3a1
5,6,0,-0.124909,-0.124908,0.67,A1,A1,4a1
6,7,0,-0.120250,-0.120250,-0.17,B2,B2,2b2
7,8,0,0.920089,0.920089,0.14,A1,A1,5a1
8,9,0,0.930130,0.930130,0.47,B2,B2,3b2
9,10,0,1.158978,1.158978,-0.37,B1,B1,2b1


In [33]:
spectrum_table(3)

Archive Table 8: N2/6-31g R=1.1  E_RHF archive=-108.867618379 ours=-108.867618373


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.718623,-15.718623,0.42,Ag,Ag,1Ag
1,2,2,-15.715262,-15.715262,0.17,B1u,B1u,1B1u
2,3,2,-1.525346,-1.525345,0.54,Ag,Ag,2Ag
3,4,2,-0.775804,-0.775804,0.17,B1u,B1u,2B1u
4,5,2,-0.629380,-0.629380,0.14,Ag,Ag,3Ag
5,6,2,-0.621235,-0.621235,-0.10,B2u,B2u,1B2u
6,7,2,-0.621235,-0.621235,-0.10,B3u,B3u,1B3u
7,8,0,0.149789,0.149789,0.05,B3g,B2g,1B3g
8,9,0,0.149789,0.149789,0.05,B2g,B3g,1B2g
9,10,0,0.588291,0.588291,0.37,B1u,B1u,3B1u


In [34]:
spectrum_table(4)

Archive Table 9: N2/6-31g R=1.8  E_RHF archive=-108.323966181 ours=-108.323966174


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.754467,-15.754467,-0.05,Ag,Ag,1Ag
1,2,2,-15.754407,-15.754407,0.19,B1u,B1u,1B1u
2,3,2,-1.107744,-1.107744,0.40,Ag,Ag,2Ag
3,4,2,-0.919264,-0.919264,-0.23,B1u,B1u,2B1u
4,5,2,-0.544720,-0.544720,-0.30,B3u,B3u,1B3u
5,6,2,-0.448838,-0.448838,-0.27,B2g,B2g,1B2g
6,7,2,-0.413060,-0.413060,-0.23,B2u,B2u,1B2u
7,8,0,-0.098146,-0.098146,-0.27,Ag,Ag,3Ag
8,9,0,-0.043452,-0.043452,0.27,B3g,B3g,1B3g
9,10,0,0.204540,0.204540,-0.37,B1u,B1u,3B1u


In [35]:
spectrum_table(5)

Archive Table 10: N2/6-31g R=3.0  E_RHF archive=-108.219724168 ours=-108.219724168


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.730667,-15.730667,-0.42,B1u,B1u,1B1u
1,2,2,-15.730635,-15.730635,0.36,Ag,Ag,1Ag
2,3,2,-0.989886,-0.989886,-0.29,Ag,Ag,2Ag
3,4,2,-0.982636,-0.982636,-0.19,B1u,B1u,2B1u
4,5,2,-0.481402,-0.481402,0.43,B3u,B2u,1B3u
5,6,2,-0.472997,-0.472997,0.18,B2g,B3g,1B2g
6,7,2,-0.295292,-0.295292,0.17,B3g,B2g,1B3g
7,8,0,-0.136367,-0.136367,0.08,B2u,B3u,1B2u
8,9,0,-0.013607,-0.013607,0.30,Ag,Ag,3Ag
9,10,0,0.053531,0.053531,0.28,B1u,B1u,3B1u


In [36]:
spectrum_table(6)

Archive Table 11: H2O/cc-pvdz R=0.9584  E_RHF archive=-76.026741428 ours=-76.026741424


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.550665,-20.550665,0.37,A1,A1,1a1
1,2,2,-1.336155,-1.336155,0.35,A1,A1,2a1
2,3,2,-0.698560,-0.698560,-0.11,B2,B2,1b2
3,4,2,-0.566489,-0.566489,0.04,A1,A1,3a1
4,5,2,-0.493087,-0.493087,0.44,B1,B1,1b1
5,6,0,0.185363,0.185363,-0.08,A1,A1,4a1
6,7,0,0.256091,0.256091,0.30,B2,B2,2b2
7,8,0,0.788274,0.788274,-0.21,B2,B2,3b2
8,9,0,0.853532,0.853532,-0.08,A1,A1,5a1
9,10,0,1.163636,1.163636,0.27,A1,A1,6a1


In [37]:
spectrum_table(7)

Archive Table 12: H2O/cc-pvdz R=1.75  E_RHF archive=-75.669624909 ours=-75.669624858


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.659582,-20.659582,0.10,A1,A1,1a1
1,2,2,-1.203329,-1.203329,0.16,A1,A1,2a1
2,3,2,-0.505358,-0.505358,0.14,B1,B1,1b1
3,4,2,-0.456938,-0.456938,-0.44,B2,B2,1b2
4,5,2,-0.442249,-0.442249,0.00,A1,A1,3a1
5,6,0,0.003215,0.003215,-0.01,A1,A1,4a1
6,7,0,0.043693,0.043692,-0.52,B2,B2,2b2
7,8,0,0.601585,0.601585,-0.20,A1,A1,5a1
8,9,0,0.641737,0.641737,0.36,B2,B2,3b2
9,10,0,1.140847,1.140847,0.00,B1,B1,2b1


In [38]:
spectrum_table(8)

Archive Table 13: H2O/cc-pvdz R=3.0  E_RHF archive=-75.430561431 ours=-75.435687184


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.705868,-20.692746,13122.44,A1,A1,1a1
1,2,2,-1.257067,-1.244594,12472.59,A1,A1,2a1
2,3,2,-0.577695,-0.563939,13755.87,A1,B1,3a1
3,4,2,-0.575160,-0.561555,13605.28,B2,B2,1b2
4,5,2,-0.268855,-0.279802,-10946.81,A1,A1,4a1
5,6,0,-0.154274,-0.160456,-6182.11,B2,B2,2b2
6,7,0,0.005498,0.013047,7549.09,B1,A1,1b1
7,8,0,0.624120,0.617649,-6471.39,A1,A1,5a1
8,9,0,0.642257,0.634104,-8153.25,B2,B2,3b2
9,10,0,1.143742,1.143208,-534.13,A1,B1,6a1


In [39]:
spectrum_table(10)

Archive Table 14: N2/cc-pvdz R=1.1  E_RHF archive=-108.953796253 ours=-108.953796241


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.687167,-15.687167,0.14,Ag,Ag,1Ag
1,2,2,-15.683810,-15.683810,-0.07,B1u,B1u,1B1u
2,3,2,-1.469497,-1.469497,0.43,Ag,Ag,2Ag
3,4,2,-0.774851,-0.774851,0.47,B1u,B1u,2B1u
4,5,2,-0.626027,-0.626026,0.51,Ag,Ag,3Ag
5,6,2,-0.607175,-0.607175,-0.35,B2u,B3u,1B2u
6,7,2,-0.607175,-0.607175,-0.35,B3u,B2u,1B3u
7,8,0,0.174428,0.174428,-0.44,B3g,B2g,1B3g
8,9,0,0.174428,0.174428,-0.43,B2g,B3g,1B2g
9,10,0,0.593893,0.593893,0.33,B1u,B1u,3B1u


In [40]:
spectrum_table(11)

Archive Table 15: N2/cc-pvdz R=1.8  E_RHF archive=-108.341640217 ours=-108.341640206


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.744033,-15.744033,-0.37,Ag,Ag,1Ag
1,2,2,-15.743960,-15.743960,0.12,B1u,B1u,1B1u
2,3,2,-1.103075,-1.103075,-0.21,Ag,Ag,2Ag
3,4,2,-0.915678,-0.915678,0.33,B1u,B1u,2B1u
4,5,2,-0.539195,-0.539195,-0.33,B3u,B3u,1B3u
5,6,2,-0.442312,-0.442312,-0.03,B2g,B2g,1B2g
6,7,2,-0.409502,-0.409502,0.29,B2u,B2u,1B2u
7,8,0,-0.093636,-0.093636,-0.32,Ag,Ag,3Ag
8,9,0,-0.036167,-0.036167,-0.39,B3g,B3g,1B3g
9,10,0,0.212900,0.212899,-0.54,B1u,B1u,3B1u


In [41]:
spectrum_table(12)

Archive Table 16: N2/cc-pvdz R=3.0  E_RHF archive=-108.231992299 ours=-108.231992299


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.721228,-15.721228,0.16,B1u,B1u,1B1u
1,2,2,-15.721202,-15.721202,0.26,Ag,Ag,1Ag
2,3,2,-0.987993,-0.987993,0.26,Ag,Ag,2Ag
3,4,2,-0.980812,-0.980812,-0.39,B1u,B1u,2B1u
4,5,2,-0.477293,-0.477293,-0.18,B3u,B2u,1B3u
5,6,2,-0.468977,-0.468977,0.01,B2g,B3g,1B2g
6,7,2,-0.291142,-0.291142,-0.49,B3g,B2g,1B3g
7,8,0,-0.131727,-0.131727,-0.35,B2u,B3u,1B2u
8,9,0,-0.008466,-0.008466,-0.40,Ag,Ag,3Ag
9,10,0,0.058459,0.058459,-0.30,B1u,B1u,3B1u


In [42]:
spectrum_table(9)

Archive Table 17: H2O/cc-pvtz R=0.95784  E_RHF archive=-76.051495438 ours=-76.057121513


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-20.548154,-20.554874,-6719.92,1a1,A1,1a1
1,2,2,-1.343878,-1.345397,-1519.04,2a1,A1,2a1
2,3,2,-0.709893,-0.709335,557.81,1b2,B2,1b2
3,4,2,-0.577841,-0.577728,113.09,3a1,A1,3a1
4,5,2,-0.504720,-0.504446,273.73,1b1,B1,1b1
5,6,0,0.141478,0.142193,715.01,4a1,A1,4a1
6,7,0,0.204005,0.204009,3.76,2b2,B2,2b2
7,8,0,0.543647,0.543864,217.36,3b2,B2,3b2
8,9,0,0.602379,0.602156,-222.87,5a1,A1,5a1
9,10,0,0.669902,0.668285,-1617.24,6a1,A1,6a1


In [43]:
spectrum_table(13)

Archive Table 18: N2/cc-pvtz R=1.09768  E_RHF archive=-108.983965064 ours=-108.983474206


,index,occ,archive_eps,ours_eps,d_uHa,archive_irrep,ours_irrep,archive_label
0,1,2,-15.681135,-15.681636,-501.27,Ag,Ag,Ag
1,2,2,-15.677599,-15.678097,-498.28,B1u,B1u,B1u
2,3,2,-1.469898,-1.469628,270.35,Ag,Ag,Ag
3,4,2,-0.777538,-0.777434,104.14,B1u,B1u,B1u
4,5,2,-0.632437,-0.632346,91.32,Ag,Ag,Ag
5,6,2,-0.612074,-0.612024,49.84,B2u,B2u,B2u
6,7,2,-0.612074,-0.612024,49.84,B3u,B3u,B3u
7,8,0,0.161879,0.161937,58.33,B2g,B2g,B2g
8,9,0,0.161879,0.161937,58.33,B3g,B3g,B3g
9,10,0,0.424762,0.425483,721.37,B1u,B1u,B1u
